# EDA v3 Main Pipeline

**Source script:** `run_eda_v3.py`

Notebook mirror for submission traceability.

In [ ]:
from __future__ import annotations

import argparse
import inspect
import json
import logging
import os
import re
import time
from itertools import combinations
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    adjusted_rand_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    normalized_mutual_info_score,
    precision_recall_fscore_support,
    roc_auc_score,
    silhouette_score,
)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    import shap
except ImportError:
    shap = None

try:
    from lightgbm import LGBMClassifier
except ImportError:
    LGBMClassifier = None

try:
    from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
except ImportError:
    dendrogram = None
    fcluster = None
    linkage = None


ROOT = Path(__file__).resolve().parent
DEFAULT_INPUT = ROOT / "outputs" / "imputed_dataset_enriched.csv"
DEFAULT_OUTDIR = ROOT / "outputs" / "eda_v3"

SEED = 42
HISTGB_SUPPORTS_CLASS_WEIGHT = "class_weight" in inspect.signature(HistGradientBoostingClassifier.__init__).parameters
BINARY_DISEASE_CLASSES = [
    "acute pancreatitis",
    "acute flare-up of triaditis/chronic enteropathy",
    "acute gastroenteritis",
    "parvovirus enteritis",
]


def setup_logging() -> logging.Logger:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
    )
    return logging.getLogger("eda_v3")


def normalize_name(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(text).lower())


def build_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def infer_outcome_column(df: pd.DataFrame) -> str:
    preferred = ["group", "diagnosis_group", "outcome_group", "outcome"]
    for col in preferred:
        if col in df.columns:
            return col
    fallback = [c for c in df.columns if "group" in normalize_name(c)]
    if fallback:
        return fallback[0]
    raise ValueError("Could not infer outcome column. Add one of: group/diagnosis_group/outcome.")


def infer_date_column(df: pd.DataFrame) -> str | None:
    preferred = ["presentation_date", "date of presentation", "presentation date"]
    for col in preferred:
        if col in df.columns:
            return col
    fallback = [c for c in df.columns if "presentation" in c.lower() and "date" in c.lower()]
    return fallback[0] if fallback else None


def to_season(month: int | float) -> str:
    if pd.isna(month):
        return "unknown"
    m = int(month)
    if m in (12, 1, 2):
        return "winter"
    if m in (3, 4, 5):
        return "spring"
    if m in (6, 7, 8):
        return "summer"
    return "autumn"


def derive_temporal_columns(df: pd.DataFrame, date_col: str | None) -> Tuple[pd.DataFrame, List[str]]:
    out = df.copy()
    temporal_cols: List[str] = []
    if date_col is None:
        return out, temporal_cols

    dt = pd.to_datetime(out[date_col], errors="coerce")
    if dt.notna().sum() == 0:
        return out, temporal_cols

    out["eda3_month"] = dt.dt.month.map(
        {
            1: "jan",
            2: "feb",
            3: "mar",
            4: "apr",
            5: "may",
            6: "jun",
            7: "jul",
            8: "aug",
            9: "sep",
            10: "oct",
            11: "nov",
            12: "dec",
        }
    ).fillna("unknown")
    out["eda3_weekday"] = dt.dt.day_name().str[:3].str.lower().fillna("unknown")
    out["eda3_season"] = dt.dt.month.map(to_season).fillna("unknown")
    temporal_cols.extend(["eda3_month", "eda3_weekday", "eda3_season"])
    return out, temporal_cols


def add_moon_cyclic_features(df: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    out = df.copy()
    created: List[str] = []
    angle_col = None
    for c in out.columns:
        if normalize_name(c) == "moonphaseangledeg":
            angle_col = c
            break
    if angle_col is None:
        return out, created

    angle = pd.to_numeric(out[angle_col], errors="coerce")
    radians = np.deg2rad(angle)
    out["moon_phase_sin"] = np.sin(radians)
    out["moon_phase_cos"] = np.cos(radians)
    created.extend(["moon_phase_sin", "moon_phase_cos"])
    return out, created


def classify_columns(
    df: pd.DataFrame, outcome_col: str, temporal_cols: Sequence[str]
) -> Dict[str, List[str]]:
    weather_keywords = [
        "temperature",
        "apparent_temperature",
        "shortwave",
        "radiation",
        "sunshine",
        "daylight",
        "precip",
        "rain",
        "snow",
        "wind",
        "et0",
        "evapo",
        "uv",
        "surface_pressure",
        "pressure",
        "sunrise",
        "sunset",
    ]
    moon_keywords = ["moon", "full_moon", "new_moon"]
    meta_patterns = [
        "id",
        "zip",
        "station",
        "town",
        "village",
        "state",
        "weather_job_id",
        "weather_missing_reason",
    ]
    leakage_patterns = [
        "suspected",
        "diagnosis",
        "clinical course",
    ]

    weather_cols: List[str] = []
    moon_cols: List[str] = []
    metadata_cols: List[str] = []
    leakage_cols: List[str] = []

    for col in df.columns:
        if col == outcome_col:
            continue
        if col in {"moon_phase_sin", "moon_phase_cos"}:

            continue
        low = col.lower()
        n = normalize_name(col)

        if any(k in low for k in moon_keywords):
            moon_cols.append(col)
            continue
        if any(k in low for k in weather_keywords):
            weather_cols.append(col)
            continue
        if any(p in low for p in leakage_patterns):
            leakage_cols.append(col)
            continue
        if any(p == n or p in low for p in meta_patterns):
            metadata_cols.append(col)
            continue

    weather_numeric = [c for c in weather_cols if pd.api.types.is_numeric_dtype(df[c])]
    weather_categorical = [c for c in weather_cols if c not in weather_numeric]

    moon_numeric = [c for c in moon_cols if pd.api.types.is_numeric_dtype(df[c])]
    moon_categorical = [c for c in moon_cols if c not in moon_numeric]

    moon_default_numeric: List[str] = []
    for c in ["moon_phase_sin", "moon_phase_cos"]:
        if c in df.columns:
            moon_default_numeric.append(c)
    for c in moon_numeric:
        if normalize_name(c) == "moonphasefractionilluminated":
            moon_default_numeric.append(c)
    moon_default_numeric = sorted(set(moon_default_numeric))

    moon_threshold_numeric = [
        c for c in moon_numeric if normalize_name(c) in {"isfullmoon", "isnewmoon"}
    ]
    moon_threshold_categorical = [
        c for c in moon_categorical if normalize_name(c) == "moonphasecategory"
    ]
    moon_mixed_numeric = sorted(set(moon_default_numeric + moon_threshold_numeric))
    moon_mixed_categorical = sorted(set(moon_threshold_categorical))

    usable_all = []
    for c in df.columns:
        if c in {outcome_col}:
            continue
        if c in set(metadata_cols) or c in set(leakage_cols):
            continue
        if c not in usable_all:
            usable_all.append(c)

    moon_original_all = set(moon_numeric + moon_categorical)
    usable_nonmoon = [c for c in usable_all if c not in moon_original_all]

    all_numeric_nonmoon = [c for c in usable_nonmoon if pd.api.types.is_numeric_dtype(df[c])]
    all_categorical_nonmoon = [c for c in usable_nonmoon if c not in all_numeric_nonmoon]
    all_numeric = sorted(set(all_numeric_nonmoon + moon_default_numeric))
    all_categorical = sorted(set(all_categorical_nonmoon))

    weather_model_numeric_base = sorted(set(weather_numeric))
    weather_model_categorical_base = sorted(set(weather_categorical + list(temporal_cols)))
    weather_model_numeric = sorted(set(weather_model_numeric_base + moon_default_numeric))
    weather_model_categorical = sorted(set(weather_model_categorical_base))

    return {
        "weather_numeric": weather_numeric,
        "weather_categorical": weather_categorical,
        "moon_numeric": moon_numeric,
        "moon_categorical": moon_categorical,
        "moon_default_numeric": moon_default_numeric,
        "moon_threshold_numeric": moon_threshold_numeric,
        "moon_threshold_categorical": moon_threshold_categorical,
        "moon_mixed_numeric": moon_mixed_numeric,
        "moon_mixed_categorical": moon_mixed_categorical,
        "metadata": metadata_cols,
        "leakage": leakage_cols,
        "all_numeric": all_numeric,
        "all_categorical": all_categorical,
        "weather_model_numeric_base": weather_model_numeric_base,
        "weather_model_categorical_base": weather_model_categorical_base,
        "weather_model_numeric": weather_model_numeric,
        "weather_model_categorical": weather_model_categorical,
    }


def build_feature_blocks(col_groups: Dict[str, List[str]]) -> Dict[str, Dict[str, List[str]]]:
    env_numeric = sorted(set(col_groups["weather_model_numeric"]))
    env_categorical = sorted(set(col_groups["weather_model_categorical"]))

    combined_numeric = sorted(set(col_groups["all_numeric"]))
    combined_categorical = sorted(set(col_groups["all_categorical"]))

    env_num_set = set(env_numeric)
    env_cat_set = set(env_categorical)
    clinical_numeric = sorted([c for c in combined_numeric if c not in env_num_set])
    clinical_categorical = sorted([c for c in combined_categorical if c not in env_cat_set])

    return {
        "clinical_only": {
            "numeric": clinical_numeric,
            "categorical": clinical_categorical,
        },
        "environmental_only": {
            "numeric": env_numeric,
            "categorical": env_categorical,
        },
        "combined": {
            "numeric": combined_numeric,
            "categorical": combined_categorical,
        },
    }


def build_feature_blocks_summary(feature_blocks: Dict[str, Dict[str, List[str]]]) -> pd.DataFrame:
    rows = []
    for block, cols in feature_blocks.items():
        n_num = len(cols["numeric"])
        n_cat = len(cols["categorical"])
        rows.append(
            {
                "feature_block": block,
                "n_numeric": n_num,
                "n_categorical": n_cat,
                "n_total": n_num + n_cat,
            }
        )
    return pd.DataFrame(rows).sort_values("feature_block").reset_index(drop=True)


def ensure_dirs(out_dir: Path) -> Dict[str, Path]:
    plots = out_dir / "plots"
    tables = out_dir / "tables"
    models = out_dir / "models"
    out_dir.mkdir(parents=True, exist_ok=True)
    plots.mkdir(parents=True, exist_ok=True)
    tables.mkdir(parents=True, exist_ok=True)
    models.mkdir(parents=True, exist_ok=True)
    return {"root": out_dir, "plots": plots, "tables": tables, "models": models}


def get_feature_names(
    preprocessor: ColumnTransformer, numeric_cols: Sequence[str], categorical_cols: Sequence[str]
) -> List[str]:
    names: List[str] = []
    if numeric_cols:
        names.extend([f"num__{c}" for c in numeric_cols])
    if categorical_cols:
        ohe = preprocessor.named_transformers_["cat"].named_steps["onehot"]
        cat_names = ohe.get_feature_names_out(categorical_cols)
        names.extend([f"cat__{c}" for c in cat_names])
    return names


def df_to_markdown_table(df: pd.DataFrame, columns: Sequence[str]) -> str:
    if df.empty:
        return "_No rows._"
    cols = [c for c in columns if c in df.columns]
    lines = []
    lines.append("| " + " | ".join(cols) + " |")
    lines.append("| " + " | ".join(["---"] * len(cols)) + " |")
    for _, row in df[cols].iterrows():
        lines.append("| " + " | ".join(str(row[c]) for c in cols) + " |")
    return "\n".join(lines)


def evaluate_and_save_model_outputs(
    name: str,
    y_test: pd.Series,
    y_pred: np.ndarray,
    labels: Sequence[str],
    out_paths: Dict[str, Path],
) -> Dict[str, float]:
    metrics = {
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_test, y_pred)),
        "macro_f1": float(f1_score(y_test, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_test, y_pred, average="weighted")),
    }

    metrics_path = out_paths["tables"] / f"{name}_metrics.json"
    metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

    cls_report = classification_report(y_test, y_pred, labels=list(labels), output_dict=True, zero_division=0)
    pd.DataFrame(cls_report).T.to_csv(out_paths["tables"] / f"{name}_classification_report.csv")

    cm = confusion_matrix(y_test, y_pred, labels=list(labels))
    cm_df = pd.DataFrame(cm, index=[f"true_{x}" for x in labels], columns=[f"pred_{x}" for x in labels])
    cm_df.to_csv(out_paths["tables"] / f"{name}_confusion_matrix.csv", index=True)

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(f"{name}: Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(out_paths["plots"] / f"{name}_confusion_matrix.png", dpi=180)
    plt.close(fig)

    return metrics


def compute_classification_metrics(y_true: pd.Series, y_pred: np.ndarray) -> Dict[str, float]:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
    }


def make_rf_pipeline(
    numeric_cols: Sequence[str],
    categorical_cols: Sequence[str],
    random_state: int,
) -> Pipeline:
    num_pipe = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )
    cat_pipe = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", build_one_hot_encoder()),
        ]
    )
    preprocessor = ColumnTransformer(
        [
            ("num", num_pipe, list(numeric_cols)),
            ("cat", cat_pipe, list(categorical_cols)),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )

    model = RandomForestClassifier(
        n_estimators=700,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=random_state,
        n_jobs=-1,
    )
    return Pipeline(
        [
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )


def make_logistic_ovr_pipeline(
    numeric_cols: Sequence[str],
    categorical_cols: Sequence[str],
    random_state: int,
) -> Pipeline:
    num_pipe = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    cat_pipe = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", build_one_hot_encoder()),
        ]
    )
    preprocessor = ColumnTransformer(
        [
            ("num", num_pipe, list(numeric_cols)),
            ("cat", cat_pipe, list(categorical_cols)),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )

    base_model = LogisticRegression(
        solver="liblinear",
        class_weight="balanced",
        max_iter=3000,
        random_state=random_state,
    )
    model = OneVsRestClassifier(base_model)
    return Pipeline(
        [
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )


def make_hist_gb_pipeline(
    numeric_cols: Sequence[str],
    categorical_cols: Sequence[str],
    random_state: int,
) -> Pipeline:
    num_pipe = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )
    cat_pipe = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", build_one_hot_encoder()),
        ]
    )
    preprocessor = ColumnTransformer(
        [
            ("num", num_pipe, list(numeric_cols)),
            ("cat", cat_pipe, list(categorical_cols)),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )

    model_kwargs: Dict[str, object] = {
        "random_state": random_state,
        "max_iter": 300,
        "learning_rate": 0.1,
        "max_depth": None,
    }
    if HISTGB_SUPPORTS_CLASS_WEIGHT:
        model_kwargs["class_weight"] = "balanced"
    model = HistGradientBoostingClassifier(**model_kwargs)
    return Pipeline(
        [
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )


def make_hist_gb_binary_pipeline(
    numeric_cols: Sequence[str],
    categorical_cols: Sequence[str],
    random_state: int,
) -> Pipeline:
    num_pipe = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )
    cat_pipe = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", build_one_hot_encoder()),
        ]
    )
    preprocessor = ColumnTransformer(
        [
            ("num", num_pipe, list(numeric_cols)),
            ("cat", cat_pipe, list(categorical_cols)),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )

    model = HistGradientBoostingClassifier(
        random_state=random_state,
        max_iter=300,
        learning_rate=0.1,
        max_depth=None,
    )
    return Pipeline(
        [
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )


def make_lgbm_binary_pipeline(
    numeric_cols: Sequence[str],
    categorical_cols: Sequence[str],
    random_state: int,
) -> Pipeline:
    if LGBMClassifier is None:
        raise ImportError("Missing dependency 'lightgbm'. Install with: pip install lightgbm")

    num_pipe = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )
    cat_pipe = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", build_one_hot_encoder()),
        ]
    )
    preprocessor = ColumnTransformer(
        [
            ("num", num_pipe, list(numeric_cols)),
            ("cat", cat_pipe, list(categorical_cols)),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )

    model = LGBMClassifier(
        random_state=random_state,
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        class_weight="balanced",
        n_jobs=1,
    )
    return Pipeline(
        [
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )


def compute_balanced_sample_weight(y: pd.Series) -> np.ndarray:
    y_series = y.astype(str)
    class_counts = y_series.value_counts()
    n = float(len(y_series))
    k = float(len(class_counts))
    weights = {cls: n / (k * cnt) for cls, cnt in class_counts.items()}
    return y_series.map(weights).to_numpy(dtype=float)


def disease_to_token(label: str) -> str:
    token = re.sub(r"[^a-z0-9]+", "_", str(label).lower()).strip("_")
    return token or "unknown"


def extract_binary_shap_matrix(shap_values: object) -> np.ndarray:
    arr: np.ndarray
    if isinstance(shap_values, list):
        if len(shap_values) == 0:
            return np.empty((0, 0), dtype=float)

        arr = np.asarray(shap_values[1] if len(shap_values) > 1 else shap_values[0], dtype=float)
    else:
        arr = np.asarray(shap_values, dtype=float)

    if arr.ndim == 3:

        out_idx = 1 if arr.shape[2] > 1 else 0
        arr = arr[:, :, out_idx]
    if arr.ndim == 1:
        arr = arr.reshape(-1, 1)
    return arr


def save_shap_bar_plot(
    shap_df: pd.DataFrame,
    disease: str,
    feature_block: str,
    out_paths: Dict[str, Path],
) -> Path:
    disease_token = disease_to_token(disease)
    block_token = disease_to_token(feature_block)
    out_file = out_paths["plots"] / f"shap_binary_{disease_token}_{block_token}.png"

    top_df = shap_df.sort_values("mean_abs_shap", ascending=False).head(20).iloc[::-1]
    fig_h = max(4.0, 0.35 * max(1, len(top_df)))
    fig, ax = plt.subplots(figsize=(10, fig_h))
    ax.barh(top_df["feature"], top_df["mean_abs_shap"], color="#4C72B0")
    ax.set_xlabel("Mean |SHAP|")
    ax.set_ylabel("Feature")
    ax.set_title(f"Binary HistGB SHAP: {disease} | {feature_block}")
    fig.tight_layout()
    fig.savefig(out_file, dpi=180)
    plt.close(fig)
    return out_file


def compute_pairwise_topk_jaccard(group_df: pd.DataFrame, k: int) -> List[float]:
    scores: List[float] = []
    fold_ids = sorted(group_df["fold"].unique().tolist())
    top_sets = {
        fold: set(group_df[(group_df["fold"] == fold) & (group_df["rank"] <= k)]["feature"].tolist())
        for fold in fold_ids
    }
    for a, b in combinations(fold_ids, 2):
        ua = top_sets[a]
        ub = top_sets[b]
        denom = len(ua | ub)
        scores.append(0.0 if denom == 0 else len(ua & ub) / denom)
    return scores


def compute_pairwise_top50_spearman(group_df: pd.DataFrame) -> List[Dict[str, object]]:
    pair_rows: List[Dict[str, object]] = []
    fold_ids = sorted(group_df["fold"].unique().tolist())
    top50 = group_df[group_df["rank"] <= 50].copy()

    top50_sets = {fold: set(top50[top50["fold"] == fold]["feature"].tolist()) for fold in fold_ids}
    rank_maps = {
        fold: top50[top50["fold"] == fold].set_index("feature")["rank"].to_dict()
        for fold in fold_ids
    }

    for a, b in combinations(fold_ids, 2):
        union_feats = sorted(top50_sets[a] | top50_sets[b])
        if not union_feats:
            spearman = np.nan
            n_feats = 0
        else:
            ra = pd.Series([float(rank_maps[a].get(f, 51.0)) for f in union_feats])
            rb = pd.Series([float(rank_maps[b].get(f, 51.0)) for f in union_feats])
            spearman = ra.corr(rb, method="spearman")
            n_feats = len(union_feats)
        pair_rows.append(
            {
                "fold_a": int(a),
                "fold_b": int(b),
                "n_union_top50_features": int(n_feats),
                "spearman_top50_union": float(spearman) if pd.notna(spearman) else np.nan,
            }
        )
    return pair_rows


def plot_spearman_matrix(
    pair_df: pd.DataFrame,
    disease: str,
    out_paths: Dict[str, Path],
) -> Path:
    disease_token = disease_to_token(disease)
    out_file = out_paths["plots"] / f"shap_stability_{disease_token}_combined.png"

    folds = sorted(set(pair_df["fold_a"].tolist()) | set(pair_df["fold_b"].tolist()))
    n = len(folds)
    mat = np.ones((n, n), dtype=float)
    fold_to_idx = {f: i for i, f in enumerate(folds)}
    for _, row in pair_df.iterrows():
        i = fold_to_idx[int(row["fold_a"])]
        j = fold_to_idx[int(row["fold_b"])]
        v = row["spearman_top50_union"]
        mat[i, j] = v
        mat[j, i] = v

    fig, ax = plt.subplots(figsize=(5.8, 5.0))
    im = ax.imshow(mat, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_title(f"SHAP Stability (Spearman)\n{disease} | combined")
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels([f"F{f}" for f in folds])
    ax.set_yticklabels([f"F{f}" for f in folds])
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f"{mat[i, j]:.2f}", ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(out_file, dpi=180)
    plt.close(fig)
    return out_file


def compute_shap_stability_outputs(
    shap_fold_df: pd.DataFrame,
    out_paths: Dict[str, Path],
    run_shap_stability: bool,
) -> Dict[str, object]:
    rankings_path = out_paths["tables"] / "shap_fold_rankings_binary.csv"
    stability_path = out_paths["tables"] / "shap_stability_summary_binary.csv"
    plot_paths: List[str] = []

    if shap_fold_df.empty:
        pd.DataFrame().to_csv(rankings_path, index=False)
        pd.DataFrame().to_csv(stability_path, index=False)
        return {
            "rankings_csv": str(rankings_path),
            "stability_csv": str(stability_path),
            "plot_files": [],
            "n_rows_rankings": 0,
            "n_rows_stability": 0,
        }

    ranked_df = shap_fold_df.copy()
    ranked_df["rank"] = ranked_df.groupby(["disease", "feature_block", "fold"])["mean_abs_shap_fold"].rank(
        method="first",
        ascending=False,
    )
    top50_rank_df = (
        ranked_df[ranked_df["rank"] <= 50]
        .sort_values(["disease", "feature_block", "fold", "rank"])
        .reset_index(drop=True)
    )
    top50_rank_df = top50_rank_df.rename(columns={"feature": "feature_name"})
    top50_rank_df = top50_rank_df[
        ["disease", "feature_block", "fold", "feature_name", "rank", "mean_abs_shap_fold"]
    ]
    top50_rank_df.to_csv(rankings_path, index=False)

    if not run_shap_stability:
        pd.DataFrame().to_csv(stability_path, index=False)
        return {
            "rankings_csv": str(rankings_path),
            "stability_csv": str(stability_path),
            "plot_files": [],
            "n_rows_rankings": int(len(top50_rank_df)),
            "n_rows_stability": 0,
        }

    stability_rows: List[Dict[str, object]] = []
    pair_rows_for_plots: List[Dict[str, object]] = []
    for (disease, block), grp in ranked_df.groupby(["disease", "feature_block"], sort=True):
        pair_rows = compute_pairwise_top50_spearman(grp)
        if pair_rows:
            pair_df = pd.DataFrame(pair_rows)
            j10 = compute_pairwise_topk_jaccard(grp, 10)
            j20 = compute_pairwise_topk_jaccard(grp, 20)
            stability_rows.append(
                {
                    "disease": disease,
                    "feature_block": block,
                    "n_folds": int(grp["fold"].nunique()),
                    "n_fold_pairs": int(len(pair_df)),
                    "spearman_mean": float(pair_df["spearman_top50_union"].mean()),
                    "spearman_std": float(pair_df["spearman_top50_union"].std(ddof=1)),
                    "jaccard_top10_mean": float(np.mean(j10)) if j10 else np.nan,
                    "jaccard_top10_std": float(np.std(j10, ddof=1)) if len(j10) > 1 else 0.0,
                    "jaccard_top20_mean": float(np.mean(j20)) if j20 else np.nan,
                    "jaccard_top20_std": float(np.std(j20, ddof=1)) if len(j20) > 1 else 0.0,
                }
            )
            if block == "combined":
                for _, r in pair_df.iterrows():
                    pair_rows_for_plots.append(
                        {
                            "disease": disease,
                            "fold_a": int(r["fold_a"]),
                            "fold_b": int(r["fold_b"]),
                            "spearman_top50_union": float(r["spearman_top50_union"]),
                        }
                    )

    stability_df = pd.DataFrame(stability_rows).sort_values(["disease", "feature_block"]).reset_index(drop=True)
    stability_df.to_csv(stability_path, index=False)

    if pair_rows_for_plots:
        pair_plot_df = pd.DataFrame(pair_rows_for_plots)
        for disease, grp in pair_plot_df.groupby("disease", sort=True):
            plot_file = plot_spearman_matrix(grp, disease=disease, out_paths=out_paths)
            plot_paths.append(str(plot_file))

    return {
        "rankings_csv": str(rankings_path),
        "stability_csv": str(stability_path),
        "plot_files": sorted(plot_paths),
        "n_rows_rankings": int(len(top50_rank_df)),
        "n_rows_stability": int(len(stability_df)),
    }


def run_envonly_permutation_test(
    df: pd.DataFrame,
    outcome_col: str,
    feature_blocks: Dict[str, Dict[str, List[str]]],
    cv_binary_summary_df: pd.DataFrame,
    out_paths: Dict[str, Path],
    random_state: int,
    n_splits: int,
    n_perm: int,
    perm_jobs: int,
    run_test: bool,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, object]]:
    dist_path = out_paths["tables"] / "permtest_envonly_auc_distribution.csv"
    summary_path = out_paths["tables"] / "permtest_envonly_auc_summary.csv"
    t0 = time.perf_counter()

    dist_cols = ["disease", "perm_idx", "perm_mean_auc"]
    summary_cols = [
        "disease",
        "observed_auc",
        "perm_mean",
        "perm_std",
        "p_value",
        "effect_size",
        "n_perm",
    ]
    if not run_test:
        pd.DataFrame(columns=dist_cols).to_csv(dist_path, index=False)
        pd.DataFrame(columns=summary_cols).to_csv(summary_path, index=False)
        return (
            pd.DataFrame(columns=dist_cols),
            pd.DataFrame(columns=summary_cols),
            {
                "enabled": False,
                "distribution_csv": str(dist_path),
                "summary_csv": str(summary_path),
                "plot_files": [],
                "perm_jobs": int(perm_jobs),
                "runtime_seconds": float(time.perf_counter() - t0),
            },
        )

    env_num = list(feature_blocks["environmental_only"]["numeric"])
    env_cat = list(feature_blocks["environmental_only"]["categorical"])
    use_cols = env_num + env_cat
    X = df[use_cols].copy()
    y_multi = df[outcome_col].astype(str)

    observed_map = {}
    if not cv_binary_summary_df.empty:
        obs_df = cv_binary_summary_df[cv_binary_summary_df["feature_block"] == "environmental_only"]
        observed_map = {
            str(r["disease"]): float(r["roc_auc_mean"])
            for _, r in obs_df.iterrows()
        }

    dist_rows: List[Dict[str, object]] = []
    summary_rows: List[Dict[str, object]] = []
    plot_files: List[str] = []

    for disease in sorted(observed_map.keys()):
        y = (y_multi == disease).astype(int)
        y_array = y.to_numpy()
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

        fixed_splits = [(train_idx, test_idx) for train_idx, test_idx in skf.split(X, y)]

        def _single_perm_auc(perm_idx: int) -> float:
            perm_seed = int(random_state + perm_idx)
            y_perm_array = np.random.default_rng(perm_seed).permutation(y_array)
            fold_aucs: List[float] = []
            for train_idx, test_idx in fixed_splits:
                X_train = X.iloc[train_idx]
                X_test = X.iloc[test_idx]
                y_train = y_perm_array[train_idx]
                y_test = y_perm_array[test_idx]

                pipe = make_hist_gb_binary_pipeline(
                    numeric_cols=env_num,
                    categorical_cols=env_cat,
                    random_state=random_state,
                )
                pipe.fit(X_train, y_train)
                y_prob = pipe.predict_proba(X_test)[:, 1]
                fold_aucs.append(float(roc_auc_score(y_test, y_prob)))
            return float(np.mean(fold_aucs))

        perm_indices = list(range(1, n_perm + 1))
        if perm_jobs == 1:
            perm_means = [_single_perm_auc(perm_idx) for perm_idx in perm_indices]
        else:
            perm_means = Parallel(n_jobs=perm_jobs, backend="loky")(
                delayed(_single_perm_auc)(perm_idx) for perm_idx in perm_indices
            )

        for perm_idx, perm_mean_auc in zip(perm_indices, perm_means):
            dist_rows.append(
                {
                    "disease": disease,
                    "perm_idx": int(perm_idx),
                    "perm_mean_auc": perm_mean_auc,
                }
            )

        observed_auc = float(observed_map[disease])
        perm_arr = np.asarray(perm_means, dtype=float)
        p_value = float((np.sum(perm_arr >= observed_auc) + 1) / (len(perm_arr) + 1))
        perm_mean = float(np.mean(perm_arr))
        summary_rows.append(
            {
                "disease": disease,
                "observed_auc": observed_auc,
                "perm_mean": perm_mean,
                "perm_std": float(np.std(perm_arr, ddof=1)),
                "p_value": p_value,
                "effect_size": float(observed_auc - perm_mean),
                "n_perm": int(n_perm),
            }
        )

        disease_token = disease_to_token(disease)
        out_plot = out_paths["plots"] / f"permtest_envonly_{disease_token}.png"
        fig, ax = plt.subplots(figsize=(7.5, 4.8))
        ax.hist(perm_arr, bins=20, color="#6B8E23", alpha=0.75, edgecolor="black")
        ax.axvline(observed_auc, color="#B22222", linestyle="--", linewidth=2, label=f"Observed AUC = {observed_auc:.3f}")
        ax.set_title(f"Permutation Test (Env-only): {disease}")
        ax.set_xlabel("Permuted mean ROC-AUC")
        ax.set_ylabel("Count")
        ax.legend(loc="best")
        fig.tight_layout()
        fig.savefig(out_plot, dpi=180)
        plt.close(fig)
        plot_files.append(str(out_plot))

    dist_df = pd.DataFrame(dist_rows, columns=dist_cols)
    if not dist_df.empty:
        dist_df = dist_df.sort_values(["disease", "perm_idx"]).reset_index(drop=True)
    summary_df = pd.DataFrame(summary_rows, columns=summary_cols)
    if not summary_df.empty:
        summary_df = summary_df.sort_values("disease").reset_index(drop=True)
    dist_df.to_csv(dist_path, index=False)
    summary_df.to_csv(summary_path, index=False)

    meta = {
        "enabled": True,
        "distribution_csv": str(dist_path),
        "summary_csv": str(summary_path),
        "plot_files": sorted(plot_files),
        "perm_jobs": int(perm_jobs),
        "runtime_seconds": float(time.perf_counter() - t0),
    }
    return dist_df, summary_df, meta


def build_robustness_summary_markdown(
    shap_stability_df: pd.DataFrame,
    perm_summary_df: pd.DataFrame,
    run_permutation_test: bool,
) -> str:
    lines: List[str] = []
    lines.append("# Robustness Summary")
    lines.append("")
    lines.append("## SHAP Stability (Binary HistGB)")
    lines.append("")
    if shap_stability_df.empty:
        lines.append("_No SHAP stability rows available._")
    else:
        show_cols = [
            "disease",
            "feature_block",
            "spearman_mean",
            "jaccard_top10_mean",
            "jaccard_top20_mean",
        ]
        tmp = shap_stability_df.copy()
        for c in ["spearman_mean", "jaccard_top10_mean", "jaccard_top20_mean"]:
            if c in tmp.columns:
                tmp[c] = tmp[c].map(lambda v: f"{v:.4f}" if pd.notna(v) else "nan")
        lines.append(df_to_markdown_table(tmp, show_cols))
    lines.append("")
    lines.append("## Permutation Test (Environmental-Only Binary HistGB)")
    lines.append("")
    if not run_permutation_test:
        lines.append("_Skipped (run with `--run-permutation-test`)._")
    elif perm_summary_df.empty:
        lines.append("_No permutation rows available._")
    else:
        tmp = perm_summary_df.copy()
        for c in ["observed_auc", "perm_mean", "perm_std", "p_value", "effect_size"]:
            if c in tmp.columns:
                tmp[c] = tmp[c].map(lambda v: f"{v:.4f}" if pd.notna(v) else "nan")
        lines.append(df_to_markdown_table(tmp, ["disease", "observed_auc", "perm_mean", "perm_std", "p_value", "effect_size"]))
    lines.append("")
    lines.append("## Note")
    lines.append("")
    lines.append(
        "These robustness checks evaluate stability/significance patterns in exploratory association modeling; they do not establish causal effects."
    )
    lines.append("")
    return "\n".join(lines)


def summarize_binary_fold_metrics(
    fold_df: pd.DataFrame,
    n_splits: int,
) -> Tuple[pd.DataFrame, Dict[str, object]]:
    summary_json: Dict[str, object] = {"n_splits": n_splits, "results": {}}
    if fold_df.empty:
        return pd.DataFrame(), summary_json

    summary_df, summary_json = summarize_binary_fold_metrics(fold_df=fold_df, n_splits=n_splits)
    return summary_df, summary_json


def run_stratified_cv_evaluation(
    df: pd.DataFrame,
    outcome_col: str,
    feature_blocks: Dict[str, Dict[str, List[str]]],
    out_paths: Dict[str, Path],
    random_state: int,
    n_splits: int,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, object]]:
    y = df[outcome_col].astype(str)
    labels = sorted(y.unique().tolist())

    model_builders = {
        "random_forest": make_rf_pipeline,
        "logistic_ovr": make_logistic_ovr_pipeline,
        "hist_gb": make_hist_gb_pipeline,
    }

    rows: List[Dict[str, object]] = []
    per_class_rows: List[Dict[str, object]] = []
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    for block_name, block_cols in feature_blocks.items():
        numeric_cols = list(block_cols["numeric"])
        categorical_cols = list(block_cols["categorical"])
        use_cols = numeric_cols + categorical_cols
        if not use_cols:
            continue

        X = df[use_cols].copy()
        for model_name, builder in model_builders.items():
            for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
                X_train = X.iloc[train_idx]
                X_test = X.iloc[test_idx]
                y_train = y.iloc[train_idx]
                y_test = y.iloc[test_idx]

                pipe = builder(
                    numeric_cols=numeric_cols,
                    categorical_cols=categorical_cols,
                    random_state=random_state,
                )
                fit_kwargs: Dict[str, object] = {}
                if model_name == "hist_gb" and not HISTGB_SUPPORTS_CLASS_WEIGHT:
                    fit_kwargs["model__sample_weight"] = compute_balanced_sample_weight(y_train)
                pipe.fit(X_train, y_train, **fit_kwargs)
                y_pred = pipe.predict(X_test)
                metrics = compute_classification_metrics(y_test, y_pred)
                precision_arr, recall_arr, f1_arr, support_arr = precision_recall_fscore_support(
                    y_test,
                    y_pred,
                    labels=labels,
                    zero_division=0,
                )

                rows.append(
                    {
                        "model": model_name,
                        "feature_block": block_name,
                        "fold": fold_idx,
                        "n_train": int(len(train_idx)),
                        "n_test": int(len(test_idx)),
                        "n_numeric_features": len(numeric_cols),
                        "n_categorical_features": len(categorical_cols),
                        **metrics,
                    }
                )
                for cls, p, r, f1v, sup in zip(labels, precision_arr, recall_arr, f1_arr, support_arr):
                    per_class_rows.append(
                        {
                            "model": model_name,
                            "feature_block": block_name,
                            "fold": fold_idx,
                            "class_label": str(cls),
                            "precision": float(p),
                            "recall": float(r),
                            "f1": float(f1v),
                            "support": int(sup),
                        }
                    )

    fold_df = pd.DataFrame(rows)
    fold_csv = out_paths["tables"] / "cv_fold_metrics.csv"
    fold_df.to_csv(fold_csv, index=False)
    per_class_fold_df = pd.DataFrame(per_class_rows)
    per_class_fold_df.to_csv(out_paths["tables"] / "cv_per_class_fold_metrics.csv", index=False)

    if fold_df.empty:
        summary_df = pd.DataFrame()
        pd.DataFrame().to_csv(out_paths["tables"] / "cv_per_class_metric_summary.csv", index=False)
        summary_json = {"n_splits": n_splits, "results": {}, "per_class_results": {}}
        (out_paths["tables"] / "cv_summary.json").write_text(
            json.dumps(summary_json, indent=2),
            encoding="utf-8",
        )
        return fold_df, summary_df, summary_json

    metric_cols = ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]
    agg_rows: List[Dict[str, object]] = []
    summary_json: Dict[str, object] = {"n_splits": n_splits, "results": {}, "per_class_results": {}}

    for (model_name, block_name), g in fold_df.groupby(["model", "feature_block"], sort=True):
        block_entry = {"n_folds": int(len(g))}
        for m in metric_cols:
            mean_v = float(g[m].mean())
            std_v = float(g[m].std(ddof=1))
            block_entry[m] = {"mean": mean_v, "std": std_v}
        summary_json["results"].setdefault(model_name, {})[block_name] = block_entry

        agg_rows.append(
            {
                "model": model_name,
                "feature_block": block_name,
                "n_folds": int(len(g)),
                "accuracy_mean": float(g["accuracy"].mean()),
                "accuracy_std": float(g["accuracy"].std(ddof=1)),
                "accuracy_mean_pm_std": f"{g['accuracy'].mean():.4f} ± {g['accuracy'].std(ddof=1):.4f}",
                "balanced_accuracy_mean": float(g["balanced_accuracy"].mean()),
                "balanced_accuracy_std": float(g["balanced_accuracy"].std(ddof=1)),
                "balanced_accuracy_mean_pm_std": f"{g['balanced_accuracy'].mean():.4f} ± {g['balanced_accuracy'].std(ddof=1):.4f}",
                "macro_f1_mean": float(g["macro_f1"].mean()),
                "macro_f1_std": float(g["macro_f1"].std(ddof=1)),
                "macro_f1_mean_pm_std": f"{g['macro_f1'].mean():.4f} ± {g['macro_f1'].std(ddof=1):.4f}",
                "weighted_f1_mean": float(g["weighted_f1"].mean()),
                "weighted_f1_std": float(g["weighted_f1"].std(ddof=1)),
                "weighted_f1_mean_pm_std": f"{g['weighted_f1'].mean():.4f} ± {g['weighted_f1'].std(ddof=1):.4f}",
            }
        )

    per_class_summary_rows: List[Dict[str, object]] = []
    if not per_class_fold_df.empty:
        for (model_name, block_name, cls), g in per_class_fold_df.groupby(
            ["model", "feature_block", "class_label"],
            sort=True,
        ):
            class_entry = {
                "n_folds": int(len(g)),
                "precision": {"mean": float(g["precision"].mean()), "std": float(g["precision"].std(ddof=1))},
                "recall": {"mean": float(g["recall"].mean()), "std": float(g["recall"].std(ddof=1))},
                "f1": {"mean": float(g["f1"].mean()), "std": float(g["f1"].std(ddof=1))},
            }
            summary_json["per_class_results"].setdefault(model_name, {}).setdefault(block_name, {})[str(cls)] = class_entry

            per_class_summary_rows.append(
                {
                    "model": model_name,
                    "feature_block": block_name,
                    "class_label": str(cls),
                    "n_folds": int(len(g)),
                    "precision_mean": float(g["precision"].mean()),
                    "precision_std": float(g["precision"].std(ddof=1)),
                    "precision_mean_pm_std": f"{g['precision'].mean():.4f} ± {g['precision'].std(ddof=1):.4f}",
                    "recall_mean": float(g["recall"].mean()),
                    "recall_std": float(g["recall"].std(ddof=1)),
                    "recall_mean_pm_std": f"{g['recall'].mean():.4f} ± {g['recall'].std(ddof=1):.4f}",
                    "f1_mean": float(g["f1"].mean()),
                    "f1_std": float(g["f1"].std(ddof=1)),
                    "f1_mean_pm_std": f"{g['f1'].mean():.4f} ± {g['f1'].std(ddof=1):.4f}",
                    "support_mean": float(g["support"].mean()),
                    "support_std": float(g["support"].std(ddof=1)),
                }
            )

    summary_df = pd.DataFrame(agg_rows).sort_values(["model", "feature_block"]).reset_index(drop=True)
    summary_df.to_csv(out_paths["tables"] / "cv_metric_summary.csv", index=False)
    per_class_summary_df = (
        pd.DataFrame(per_class_summary_rows)
        .sort_values(["model", "feature_block", "class_label"])
        .reset_index(drop=True)
    )
    per_class_summary_df.to_csv(out_paths["tables"] / "cv_per_class_metric_summary.csv", index=False)
    (out_paths["tables"] / "cv_summary.json").write_text(
        json.dumps(summary_json, indent=2),
        encoding="utf-8",
    )
    return fold_df, summary_df, summary_json


def run_binary_histgb_cv(
    df: pd.DataFrame,
    outcome_col: str,
    feature_blocks: Dict[str, Dict[str, List[str]]],
    out_paths: Dict[str, Path],
    random_state: int,
    n_splits: int,
    disease_classes: Sequence[str] = BINARY_DISEASE_CLASSES,
    run_shap_stability: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, object], Dict[str, object]]:
    if shap is None:
        raise ImportError("Missing dependency 'shap'. Install with: pip install shap")

    t0 = time.perf_counter()
    y_multiclass = df[outcome_col].astype(str)
    available = y_multiclass.unique().tolist()
    available_by_norm = {normalize_name(a): a for a in available}
    selected_diseases = []
    for disease in disease_classes:
        resolved = available_by_norm.get(normalize_name(disease))
        if resolved is not None:
            selected_diseases.append(resolved)
    selected_diseases = sorted(set(selected_diseases))

    fold_rows: List[Dict[str, object]] = []
    shap_fold_rows: List[Dict[str, object]] = []
    summary_json: Dict[str, object] = {"n_splits": n_splits, "results": {}}
    shap_warnings: List[str] = []
    shap_plot_paths: List[str] = []
    shap_summary_path = out_paths["tables"] / "shap_summary_binary.csv"
    shap_top20_path = out_paths["tables"] / "shap_top20_binary.csv"

    for disease in selected_diseases:
        y_bin = (y_multiclass == disease).astype(int)
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        model_name = f"hist_gb_binary_{disease_to_token(disease)}"

        for block_name, block_cols in feature_blocks.items():
            numeric_cols = list(block_cols["numeric"])
            categorical_cols = list(block_cols["categorical"])
            use_cols = numeric_cols + categorical_cols
            if not use_cols:
                continue

            X = df[use_cols].copy()
            for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y_bin), start=1):
                X_train = X.iloc[train_idx]
                X_test = X.iloc[test_idx]
                y_train = y_bin.iloc[train_idx]
                y_test = y_bin.iloc[test_idx]

                pipe = make_hist_gb_binary_pipeline(
                    numeric_cols=numeric_cols,
                    categorical_cols=categorical_cols,
                    random_state=random_state,
                )
                pipe.fit(X_train, y_train)
                y_pred = pipe.predict(X_test)
                y_prob = pipe.predict_proba(X_test)[:, 1]


                try:
                    preprocessor = pipe.named_steps["preprocessor"]
                    model = pipe.named_steps["model"]
                    X_val_trans = preprocessor.transform(X_test)
                    feature_names = get_feature_names(preprocessor, numeric_cols, categorical_cols)

                    explainer = shap.TreeExplainer(model)
                    shap_raw = explainer.shap_values(X_val_trans)
                    shap_mat = extract_binary_shap_matrix(shap_raw)
                    if shap_mat.shape[0] == 0 or shap_mat.shape[1] == 0:
                        raise ValueError("TreeExplainer returned empty SHAP matrix.")

                    if shap_mat.shape[1] != len(feature_names):
                        raise ValueError(
                            f"SHAP feature mismatch: {shap_mat.shape[1]} SHAP columns vs {len(feature_names)} names."
                        )

                    mean_abs_fold = np.abs(shap_mat).mean(axis=0)
                    for feature_name, mean_abs in zip(feature_names, mean_abs_fold):
                        shap_fold_rows.append(
                            {
                                "disease": disease,
                                "feature_block": block_name,
                                "fold": fold_idx,
                                "feature": feature_name,
                                "mean_abs_shap_fold": float(mean_abs),
                            }
                        )
                except Exception as exc:
                    shap_warnings.append(
                        f"SHAP failed for disease='{disease}', block='{block_name}', fold={fold_idx}: {exc}"
                    )

                p, r, f1v, _ = precision_recall_fscore_support(
                    y_test,
                    y_pred,
                    average="binary",
                    zero_division=0,
                )
                row = {
                    "model": model_name,
                    "disease": disease,
                    "feature_block": block_name,
                    "fold": fold_idx,
                    "n_train": int(len(train_idx)),
                    "n_test": int(len(test_idx)),
                    "n_positive_train": int(y_train.sum()),
                    "n_positive_test": int(y_test.sum()),
                    "n_numeric_features": len(numeric_cols),
                    "n_categorical_features": len(categorical_cols),
                    "balanced_accuracy": float(balanced_accuracy_score(y_test, y_pred)),
                    "recall": float(r),
                    "precision": float(p),
                    "f1": float(f1v),
                    "roc_auc": float(roc_auc_score(y_test, y_prob)),
                }
                fold_rows.append(row)

    fold_df = pd.DataFrame(fold_rows)
    fold_df.to_csv(out_paths["tables"] / "cv_binary_fold_metrics.csv", index=False)

    if fold_df.empty:
        summary_df = pd.DataFrame()
        summary_df.to_csv(out_paths["tables"] / "cv_binary_metric_summary.csv", index=False)
        (out_paths["tables"] / "cv_binary_summary.json").write_text(
            json.dumps(summary_json, indent=2),
            encoding="utf-8",
        )
        pd.DataFrame().to_csv(shap_summary_path, index=False)
        pd.DataFrame().to_csv(shap_top20_path, index=False)
        shap_stability_meta = compute_shap_stability_outputs(
            shap_fold_df=pd.DataFrame(),
            out_paths=out_paths,
            run_shap_stability=run_shap_stability,
        )
        shap_meta = {
            "summary_csv": str(shap_summary_path),
            "top20_csv": str(shap_top20_path),
            "plot_files": [],
            "warnings": shap_warnings,
            "stability": shap_stability_meta,
            "runtime_seconds": float(time.perf_counter() - t0),
        }
        return fold_df, summary_df, summary_json, shap_meta

    metric_cols = ["balanced_accuracy", "recall", "precision", "f1", "roc_auc"]
    summary_rows: List[Dict[str, object]] = []
    for (model_name, disease, block_name), g in fold_df.groupby(
        ["model", "disease", "feature_block"], sort=True
    ):
        block_entry: Dict[str, object] = {"n_folds": int(len(g))}
        for m in metric_cols:
            block_entry[m] = {"mean": float(g[m].mean()), "std": float(g[m].std(ddof=1))}

        summary_json["results"].setdefault(disease, {})[block_name] = {
            "model": model_name,
            **block_entry,
        }

        summary_rows.append(
            {
                "model": model_name,
                "disease": disease,
                "feature_block": block_name,
                "n_folds": int(len(g)),
                "balanced_accuracy_mean": float(g["balanced_accuracy"].mean()),
                "balanced_accuracy_std": float(g["balanced_accuracy"].std(ddof=1)),
                "balanced_accuracy_mean_pm_std": f"{g['balanced_accuracy'].mean():.4f} ± {g['balanced_accuracy'].std(ddof=1):.4f}",
                "recall_mean": float(g["recall"].mean()),
                "recall_std": float(g["recall"].std(ddof=1)),
                "recall_mean_pm_std": f"{g['recall'].mean():.4f} ± {g['recall'].std(ddof=1):.4f}",
                "precision_mean": float(g["precision"].mean()),
                "precision_std": float(g["precision"].std(ddof=1)),
                "precision_mean_pm_std": f"{g['precision'].mean():.4f} ± {g['precision'].std(ddof=1):.4f}",
                "f1_mean": float(g["f1"].mean()),
                "f1_std": float(g["f1"].std(ddof=1)),
                "f1_mean_pm_std": f"{g['f1'].mean():.4f} ± {g['f1'].std(ddof=1):.4f}",
                "roc_auc_mean": float(g["roc_auc"].mean()),
                "roc_auc_std": float(g["roc_auc"].std(ddof=1)),
                "roc_auc_mean_pm_std": f"{g['roc_auc'].mean():.4f} ± {g['roc_auc'].std(ddof=1):.4f}",
            }
        )

    summary_df = (
        pd.DataFrame(summary_rows)
        .sort_values(["disease", "feature_block"])
        .reset_index(drop=True)
    )
    summary_df.to_csv(out_paths["tables"] / "cv_binary_metric_summary.csv", index=False)
    (out_paths["tables"] / "cv_binary_summary.json").write_text(
        json.dumps(summary_json, indent=2),
        encoding="utf-8",
    )
    shap_fold_df = pd.DataFrame(shap_fold_rows)
    if shap_fold_df.empty:
        shap_summary_df = pd.DataFrame()
        shap_top20_df = pd.DataFrame()
    else:
        shap_summary_df = (
            shap_fold_df.groupby(["disease", "feature_block", "feature"], as_index=False)
            .agg(
                mean_abs_shap=("mean_abs_shap_fold", "mean"),
                std_abs_shap=("mean_abs_shap_fold", "std"),
                n_folds=("fold", "nunique"),
            )
        )
        shap_summary_df["std_abs_shap"] = shap_summary_df["std_abs_shap"].fillna(0.0)
        shap_summary_df = shap_summary_df.sort_values(
            ["disease", "feature_block", "mean_abs_shap"], ascending=[True, True, False]
        ).reset_index(drop=True)

        shap_top20_df = (
            shap_summary_df.groupby(["disease", "feature_block"], as_index=False, group_keys=False)
            .head(20)
            .copy()
            .reset_index(drop=True)
        )
        shap_top20_df["rank_within_group"] = (
            shap_top20_df.groupby(["disease", "feature_block"]).cumcount() + 1
        )
        shap_top20_df = shap_top20_df[
            [
                "disease",
                "feature_block",
                "rank_within_group",
                "feature",
                "mean_abs_shap",
                "std_abs_shap",
            ]
        ]

        for (disease, block_name), grp in shap_summary_df.groupby(["disease", "feature_block"], sort=True):
            plot_path = save_shap_bar_plot(
                shap_df=grp,
                disease=disease,
                feature_block=block_name,
                out_paths=out_paths,
            )
            shap_plot_paths.append(str(plot_path))

    shap_summary_df.to_csv(shap_summary_path, index=False)
    shap_top20_df.to_csv(shap_top20_path, index=False)
    shap_stability_meta = compute_shap_stability_outputs(
        shap_fold_df=shap_fold_df,
        out_paths=out_paths,
        run_shap_stability=run_shap_stability,
    )
    shap_meta = {
        "summary_csv": str(shap_summary_path),
        "top20_csv": str(shap_top20_path),
        "plot_files": sorted(shap_plot_paths),
        "warnings": shap_warnings,
        "stability": shap_stability_meta,
        "runtime_seconds": float(time.perf_counter() - t0),
    }
    return fold_df, summary_df, summary_json, shap_meta


def run_binary_lgbm_cv(
    df: pd.DataFrame,
    outcome_col: str,
    feature_blocks: Dict[str, Dict[str, List[str]]],
    out_paths: Dict[str, Path],
    random_state: int,
    n_splits: int,
    disease_classes: Sequence[str] = BINARY_DISEASE_CLASSES,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, object], Dict[str, object]]:
    if LGBMClassifier is None:
        raise ImportError("Missing dependency 'lightgbm'. Install with: pip install lightgbm")

    t0 = time.perf_counter()
    y_multiclass = df[outcome_col].astype(str)
    available = y_multiclass.unique().tolist()
    available_by_norm = {normalize_name(a): a for a in available}
    selected_diseases = []
    for disease in disease_classes:
        resolved = available_by_norm.get(normalize_name(disease))
        if resolved is not None:
            selected_diseases.append(resolved)
    selected_diseases = sorted(set(selected_diseases))

    fold_rows: List[Dict[str, object]] = []
    shap_fold_rows: List[Dict[str, object]] = []
    shap_warnings: List[str] = []
    shap_summary_path = out_paths["tables"] / "shap_summary_binary_lgbm.csv"

    for disease in selected_diseases:
        y_bin = (y_multiclass == disease).astype(int)
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        model_name = f"lgbm_binary_{disease_to_token(disease)}"

        for block_name, block_cols in feature_blocks.items():
            numeric_cols = list(block_cols["numeric"])
            categorical_cols = list(block_cols["categorical"])
            use_cols = numeric_cols + categorical_cols
            if not use_cols:
                continue

            X = df[use_cols].copy()
            for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y_bin), start=1):
                X_train = X.iloc[train_idx]
                X_test = X.iloc[test_idx]
                y_train = y_bin.iloc[train_idx]
                y_test = y_bin.iloc[test_idx]

                pipe = make_lgbm_binary_pipeline(
                    numeric_cols=numeric_cols,
                    categorical_cols=categorical_cols,
                    random_state=random_state,
                )
                pipe.fit(X_train, y_train)
                y_pred = pipe.predict(X_test)
                y_prob = pipe.predict_proba(X_test)[:, 1]

                if shap is not None:
                    try:
                        preprocessor = pipe.named_steps["preprocessor"]
                        model = pipe.named_steps["model"]
                        X_val_trans = preprocessor.transform(X_test)
                        feature_names = get_feature_names(preprocessor, numeric_cols, categorical_cols)

                        explainer = shap.TreeExplainer(model)
                        shap_raw = explainer.shap_values(X_val_trans)
                        shap_mat = extract_binary_shap_matrix(shap_raw)
                        if shap_mat.shape[1] != len(feature_names):
                            raise ValueError(
                                f"SHAP feature mismatch: {shap_mat.shape[1]} SHAP columns vs {len(feature_names)} names."
                            )
                        mean_abs_fold = np.abs(shap_mat).mean(axis=0)
                        for feature_name, mean_abs in zip(feature_names, mean_abs_fold):
                            shap_fold_rows.append(
                                {
                                    "disease": disease,
                                    "feature_block": block_name,
                                    "fold": fold_idx,
                                    "feature": feature_name,
                                    "mean_abs_shap_fold": float(mean_abs),
                                }
                            )
                    except Exception as exc:
                        shap_warnings.append(
                            f"LightGBM SHAP failed for disease='{disease}', block='{block_name}', fold={fold_idx}: {exc}"
                        )

                p, r, f1v, _ = precision_recall_fscore_support(
                    y_test,
                    y_pred,
                    average="binary",
                    zero_division=0,
                )
                fold_rows.append(
                    {
                        "model": model_name,
                        "disease": disease,
                        "feature_block": block_name,
                        "fold": fold_idx,
                        "n_train": int(len(train_idx)),
                        "n_test": int(len(test_idx)),
                        "n_positive_train": int(y_train.sum()),
                        "n_positive_test": int(y_test.sum()),
                        "n_numeric_features": len(numeric_cols),
                        "n_categorical_features": len(categorical_cols),
                        "balanced_accuracy": float(balanced_accuracy_score(y_test, y_pred)),
                        "recall": float(r),
                        "precision": float(p),
                        "f1": float(f1v),
                        "roc_auc": float(roc_auc_score(y_test, y_prob)),
                    }
                )

    fold_df = pd.DataFrame(fold_rows)
    fold_df.to_csv(out_paths["tables"] / "cv_binary_fold_metrics_lgbm.csv", index=False)

    summary_df, summary_json = summarize_binary_fold_metrics(fold_df=fold_df, n_splits=n_splits)
    summary_df.to_csv(out_paths["tables"] / "cv_binary_metric_summary_lgbm.csv", index=False)

    shap_meta: Dict[str, object] = {
        "enabled": bool(shap is not None),
        "summary_csv": "",
        "warnings": shap_warnings,
        "runtime_seconds": float(time.perf_counter() - t0),
    }
    if shap is None:
        shap_warnings.append("SHAP unavailable; skipped LightGBM SHAP summary.")
    else:
        if shap_fold_rows:
            shap_fold_df = pd.DataFrame(shap_fold_rows)
            shap_summary_df = (
                shap_fold_df.groupby(["disease", "feature_block", "feature"], as_index=False)
                .agg(
                    mean_abs_shap=("mean_abs_shap_fold", "mean"),
                    std_abs_shap=("mean_abs_shap_fold", "std"),
                    n_folds=("fold", "nunique"),
                )
                .sort_values(["disease", "feature_block", "mean_abs_shap"], ascending=[True, True, False])
                .reset_index(drop=True)
            )
            shap_summary_df["std_abs_shap"] = shap_summary_df["std_abs_shap"].fillna(0.0)
        else:
            shap_summary_df = pd.DataFrame(
                columns=["disease", "feature_block", "feature", "mean_abs_shap", "std_abs_shap", "n_folds"]
            )
        shap_summary_df.to_csv(shap_summary_path, index=False)
        shap_meta["summary_csv"] = str(shap_summary_path)

    return fold_df, summary_df, summary_json, shap_meta


def run_envonly_permutation_test_lgbm(
    df: pd.DataFrame,
    outcome_col: str,
    feature_blocks: Dict[str, Dict[str, List[str]]],
    cv_binary_summary_df: pd.DataFrame,
    out_paths: Dict[str, Path],
    random_state: int,
    n_splits: int,
    n_perm: int,
    perm_jobs: int,
    run_test: bool,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, object]]:
    if LGBMClassifier is None:
        raise ImportError("Missing dependency 'lightgbm'. Install with: pip install lightgbm")

    dist_path = out_paths["tables"] / "permtest_envonly_auc_distribution_lgbm.csv"
    summary_path = out_paths["tables"] / "permtest_envonly_auc_summary_lgbm.csv"
    t0 = time.perf_counter()

    dist_cols = ["disease", "perm_idx", "perm_mean_auc"]
    summary_cols = [
        "disease",
        "observed_auc",
        "perm_mean",
        "perm_std",
        "p_value",
        "effect_size",
        "n_perm",
    ]
    if not run_test:
        pd.DataFrame(columns=dist_cols).to_csv(dist_path, index=False)
        pd.DataFrame(columns=summary_cols).to_csv(summary_path, index=False)
        return (
            pd.DataFrame(columns=dist_cols),
            pd.DataFrame(columns=summary_cols),
            {
                "enabled": False,
                "distribution_csv": str(dist_path),
                "summary_csv": str(summary_path),
                "plot_files": [],
                "perm_jobs": int(perm_jobs),
                "runtime_seconds": float(time.perf_counter() - t0),
            },
        )

    env_num = list(feature_blocks["environmental_only"]["numeric"])
    env_cat = list(feature_blocks["environmental_only"]["categorical"])
    use_cols = env_num + env_cat
    X = df[use_cols].copy()
    y_multi = df[outcome_col].astype(str)

    observed_map = {}
    if not cv_binary_summary_df.empty:
        obs_df = cv_binary_summary_df[cv_binary_summary_df["feature_block"] == "environmental_only"]
        observed_map = {
            str(r["disease"]): float(r["roc_auc_mean"])
            for _, r in obs_df.iterrows()
        }

    dist_rows: List[Dict[str, object]] = []
    summary_rows: List[Dict[str, object]] = []
    plot_files: List[str] = []

    for disease in sorted(observed_map.keys()):
        y = (y_multi == disease).astype(int)
        y_array = y.to_numpy()
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        fixed_splits = [(train_idx, test_idx) for train_idx, test_idx in skf.split(X, y)]

        def _single_perm_auc(perm_idx: int) -> float:
            perm_seed = int(random_state + perm_idx)
            y_perm_array = np.random.default_rng(perm_seed).permutation(y_array)
            fold_aucs: List[float] = []
            for train_idx, test_idx in fixed_splits:
                X_train = X.iloc[train_idx]
                X_test = X.iloc[test_idx]
                y_train = y_perm_array[train_idx]
                y_test = y_perm_array[test_idx]

                pipe = make_lgbm_binary_pipeline(
                    numeric_cols=env_num,
                    categorical_cols=env_cat,
                    random_state=random_state,
                )
                pipe.fit(X_train, y_train)
                y_prob = pipe.predict_proba(X_test)[:, 1]
                fold_aucs.append(float(roc_auc_score(y_test, y_prob)))
            return float(np.mean(fold_aucs))

        perm_indices = list(range(1, n_perm + 1))
        if perm_jobs == 1:
            perm_means = [_single_perm_auc(perm_idx) for perm_idx in perm_indices]
        else:
            perm_means = Parallel(n_jobs=perm_jobs, backend="loky")(
                delayed(_single_perm_auc)(perm_idx) for perm_idx in perm_indices
            )

        for perm_idx, perm_mean_auc in zip(perm_indices, perm_means):
            dist_rows.append(
                {
                    "disease": disease,
                    "perm_idx": int(perm_idx),
                    "perm_mean_auc": perm_mean_auc,
                }
            )

        observed_auc = float(observed_map[disease])
        perm_arr = np.asarray(perm_means, dtype=float)
        p_value = float((np.sum(perm_arr >= observed_auc) + 1) / (len(perm_arr) + 1))
        perm_mean = float(np.mean(perm_arr))
        summary_rows.append(
            {
                "disease": disease,
                "observed_auc": observed_auc,
                "perm_mean": perm_mean,
                "perm_std": float(np.std(perm_arr, ddof=1)),
                "p_value": p_value,
                "effect_size": float(observed_auc - perm_mean),
                "n_perm": int(n_perm),
            }
        )

        disease_token = disease_to_token(disease)
        out_plot = out_paths["plots"] / f"permtest_envonly_{disease_token}_lgbm.png"
        fig, ax = plt.subplots(figsize=(7.5, 4.8))
        ax.hist(perm_arr, bins=20, color="#1F77B4", alpha=0.75, edgecolor="black")
        ax.axvline(observed_auc, color="#B22222", linestyle="--", linewidth=2, label=f"Observed AUC = {observed_auc:.3f}")
        ax.set_title(f"Permutation Test (Env-only LightGBM): {disease}")
        ax.set_xlabel("Permuted mean ROC-AUC")
        ax.set_ylabel("Count")
        ax.legend(loc="best")
        fig.tight_layout()
        fig.savefig(out_plot, dpi=180)
        plt.close(fig)
        plot_files.append(str(out_plot))

    dist_df = pd.DataFrame(dist_rows, columns=dist_cols)
    if not dist_df.empty:
        dist_df = dist_df.sort_values(["disease", "perm_idx"]).reset_index(drop=True)
    summary_df = pd.DataFrame(summary_rows, columns=summary_cols)
    if not summary_df.empty:
        summary_df = summary_df.sort_values("disease").reset_index(drop=True)
    dist_df.to_csv(dist_path, index=False)
    summary_df.to_csv(summary_path, index=False)

    meta = {
        "enabled": True,
        "distribution_csv": str(dist_path),
        "summary_csv": str(summary_path),
        "plot_files": sorted(plot_files),
        "perm_jobs": int(perm_jobs),
        "runtime_seconds": float(time.perf_counter() - t0),
    }
    return dist_df, summary_df, meta


def build_binary_model_robustness_comparison(
    hist_cv_summary_df: pd.DataFrame,
    hist_perm_summary_df: pd.DataFrame,
    lgbm_cv_summary_df: pd.DataFrame,
    lgbm_perm_summary_df: pd.DataFrame,
) -> pd.DataFrame:
    hist_auc = hist_cv_summary_df[hist_cv_summary_df["feature_block"] == "environmental_only"][
        ["disease", "roc_auc_mean"]
    ].rename(columns={"roc_auc_mean": "hist_observed_auc"})
    lgbm_auc = lgbm_cv_summary_df[lgbm_cv_summary_df["feature_block"] == "environmental_only"][
        ["disease", "roc_auc_mean"]
    ].rename(columns={"roc_auc_mean": "lgbm_observed_auc"})
    hist_p = hist_perm_summary_df[["disease", "p_value"]].rename(columns={"p_value": "hist_perm_p_value"})
    lgbm_p = lgbm_perm_summary_df[["disease", "p_value"]].rename(columns={"p_value": "lgbm_perm_p_value"})

    out = hist_auc.merge(lgbm_auc, on="disease", how="outer")
    out = out.merge(hist_p, on="disease", how="left")
    out = out.merge(lgbm_p, on="disease", how="left")

    out["hist_robust"] = out["hist_perm_p_value"] < 0.05
    out["lgbm_robust"] = out["lgbm_perm_p_value"] < 0.05
    have_p = out["hist_perm_p_value"].notna() & out["lgbm_perm_p_value"].notna()
    out["robustness_agreement"] = np.where(
        have_p & (out["hist_robust"] == out["lgbm_robust"]),
        "agree",
        np.where(have_p, "disagree", "insufficient_permutation_data"),
    )
    return out.sort_values("disease").reset_index(drop=True)


def run_random_forest(
    df: pd.DataFrame,
    outcome_col: str,
    numeric_cols: Sequence[str],
    categorical_cols: Sequence[str],
    out_paths: Dict[str, Path],
    run_name: str,
    random_state: int,
    test_size: float,
) -> Tuple[Dict[str, float], pd.DataFrame]:
    X = df[list(numeric_cols) + list(categorical_cols)].copy()
    y = df[outcome_col].astype(str)

    labels = sorted(y.unique().tolist())
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y,
    )

    num_pipe = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )
    cat_pipe = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", build_one_hot_encoder()),
        ]
    )
    preprocessor = ColumnTransformer(
        [
            ("num", num_pipe, list(numeric_cols)),
            ("cat", cat_pipe, list(categorical_cols)),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )

    model = RandomForestClassifier(
        n_estimators=700,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=random_state,
        n_jobs=-1,
    )

    pipe = Pipeline(
        [
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )
    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    metrics = evaluate_and_save_model_outputs(run_name, y_test, y_pred, labels, out_paths)

    fitted_pre = pipe.named_steps["preprocessor"]
    fitted_model = pipe.named_steps["model"]
    feature_names = get_feature_names(fitted_pre, numeric_cols, categorical_cols)
    importances = fitted_model.feature_importances_

    imp_df = pd.DataFrame({"feature": feature_names, "importance": importances})
    imp_df = imp_df.sort_values("importance", ascending=False).reset_index(drop=True)
    imp_df.to_csv(out_paths["tables"] / f"{run_name}_feature_importance.csv", index=False)

    top = imp_df.head(25).iloc[::-1]
    fig, ax = plt.subplots(figsize=(11, 8))
    ax.barh(top["feature"], top["importance"], color="#2B6EA6")
    ax.set_title(f"{run_name}: Top 25 Feature Importances")
    ax.set_xlabel("Importance")
    ax.set_ylabel("Feature")
    fig.tight_layout()
    fig.savefig(out_paths["plots"] / f"{run_name}_feature_importance_top25.png", dpi=180)
    plt.close(fig)

    return metrics, imp_df


def run_pca_weather(
    df: pd.DataFrame,
    outcome_col: str,
    weather_numeric_cols: Sequence[str],
    out_paths: Dict[str, Path],
    n_components_max: int = 20,
) -> pd.DataFrame:
    if not weather_numeric_cols:
        return pd.DataFrame()

    X = df[list(weather_numeric_cols)].copy()
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.median(numeric_only=True))

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    n_components = min(n_components_max, X_scaled.shape[1], X_scaled.shape[0] - 1)
    pca = PCA(n_components=n_components, random_state=SEED)
    scores = pca.fit_transform(X_scaled)

    evr = pca.explained_variance_ratio_
    evr_df = pd.DataFrame(
        {
            "component": [f"PC{i}" for i in range(1, len(evr) + 1)],
            "explained_variance_ratio": evr,
            "cumulative_variance_ratio": np.cumsum(evr),
        }
    )
    evr_df.to_csv(out_paths["tables"] / "pca_explained_variance.csv", index=False)

    fig, ax1 = plt.subplots(figsize=(9, 5))
    x = np.arange(1, len(evr) + 1)
    ax1.bar(x, evr, color="#4F81BD", alpha=0.8)
    ax1.plot(x, np.cumsum(evr), color="#C0504D", marker="o")
    ax1.set_title("PCA Scree Plot (Weather Features)")
    ax1.set_xlabel("Principal Component")
    ax1.set_ylabel("Explained Variance Ratio")
    ax1.set_xticks(x)
    ax1.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(out_paths["plots"] / "pca_scree_plot.png", dpi=180)
    plt.close(fig)

    loading = pd.DataFrame(
        pca.components_.T,
        index=weather_numeric_cols,
        columns=[f"PC{i}" for i in range(1, len(evr) + 1)],
    )
    loading.to_csv(out_paths["tables"] / "pca_loadings_full.csv", index=True)

    top_rows: List[pd.DataFrame] = []
    for pc in ["PC1", "PC2", "PC3"]:
        if pc not in loading.columns:
            continue
        s = loading[pc].abs().sort_values(ascending=False).head(20)
        d = pd.DataFrame(
            {
                "pc": pc,
                "feature": s.index,
                "loading_abs": s.values,
                "loading_signed": loading.loc[s.index, pc].values,
            }
        )
        top_rows.append(d)
    top_df = pd.concat(top_rows, axis=0, ignore_index=True) if top_rows else pd.DataFrame()
    top_df.to_csv(out_paths["tables"] / "pca_top_loadings_pc1_pc3.csv", index=False)

    score_cols = [f"PC{i}" for i in range(1, scores.shape[1] + 1)]
    score_df = pd.DataFrame(scores, columns=score_cols)
    score_df[outcome_col] = df[outcome_col].astype(str).values
    score_df.to_csv(out_paths["tables"] / "pca_scores.csv", index=False)

    if {"PC1", "PC2"}.issubset(score_df.columns):
        plot_pca_scatter(
            score_df=score_df,
            x_col="PC1",
            y_col="PC2",
            outcome_col=outcome_col,
            out_file=out_paths["plots"] / "pca_pc1_pc2_by_group.png",
            title="PCA: PC1 vs PC2 by Group",
        )
    if {"PC2", "PC3"}.issubset(score_df.columns):
        plot_pca_scatter(
            score_df=score_df,
            x_col="PC2",
            y_col="PC3",
            outcome_col=outcome_col,
            out_file=out_paths["plots"] / "pca_pc2_pc3_by_group.png",
            title="PCA: PC2 vs PC3 by Group",
        )

    return evr_df


def plot_pca_scatter(
    score_df: pd.DataFrame,
    x_col: str,
    y_col: str,
    outcome_col: str,
    out_file: Path,
    title: str,
) -> None:
    fig, ax = plt.subplots(figsize=(8, 6))
    classes = sorted(score_df[outcome_col].astype(str).unique().tolist())
    colors = plt.cm.tab10(np.linspace(0, 1, len(classes)))
    for cls, color in zip(classes, colors):
        sub = score_df[score_df[outcome_col].astype(str) == cls]
        ax.scatter(sub[x_col], sub[y_col], s=26, alpha=0.75, color=color, label=cls)
    ax.set_title(title)
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8, loc="best")
    fig.tight_layout()
    fig.savefig(out_file, dpi=180)
    plt.close(fig)


def ensure_unsupervised_dirs(base_outdir: Path) -> Dict[str, Path]:
    root = base_outdir / "unsupervised"
    tables = root / "tables"
    plots = root / "plots"
    root.mkdir(parents=True, exist_ok=True)
    tables.mkdir(parents=True, exist_ok=True)
    plots.mkdir(parents=True, exist_ok=True)
    return {"root": root, "tables": tables, "plots": plots}


def build_numeric_matrix(df: pd.DataFrame, numeric_cols: Sequence[str]) -> pd.DataFrame:
    if not numeric_cols:
        return pd.DataFrame(index=df.index)
    X = df[list(numeric_cols)].copy()
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.median(numeric_only=True))
    return X


def compute_pc_threshold(cumulative: np.ndarray, threshold: float) -> int:
    if cumulative.size == 0:
        return 0
    idx = int(np.searchsorted(cumulative, threshold, side="left"))
    return min(len(cumulative), idx + 1)


def save_contingency_with_percentages(
    cluster_labels: Sequence[int],
    diagnosis_labels: Sequence[str],
    out_file: Path,
) -> pd.DataFrame:
    ctab = pd.crosstab(
        pd.Series(cluster_labels, name="cluster"),
        pd.Series(diagnosis_labels, name="diagnosis"),
        dropna=False,
    )
    if ctab.empty:
        out = pd.DataFrame(columns=["cluster", "diagnosis", "count", "row_pct", "col_pct"])
        out.to_csv(out_file, index=False)
        return out

    row_pct = ctab.div(ctab.sum(axis=1), axis=0).fillna(0.0) * 100.0
    col_pct = ctab.div(ctab.sum(axis=0), axis=1).fillna(0.0) * 100.0
    rows: List[Dict[str, object]] = []
    for cl in ctab.index:
        for dx in ctab.columns:
            rows.append(
                {
                    "cluster": int(cl),
                    "diagnosis": str(dx),
                    "count": int(ctab.loc[cl, dx]),
                    "row_pct": float(row_pct.loc[cl, dx]),
                    "col_pct": float(col_pct.loc[cl, dx]),
                }
            )
    out = pd.DataFrame(rows).sort_values(["cluster", "diagnosis"]).reset_index(drop=True)
    out.to_csv(out_file, index=False)
    return out


def plot_kmeans_clusters_pc12(
    scores: np.ndarray,
    cluster_labels: Sequence[int],
    block: str,
    dtag: str,
    k_label: str,
    out_path: Path,
) -> None:
    if scores.shape[1] < 2:
        return
    labels_arr = np.asarray(cluster_labels)
    fig, ax = plt.subplots(figsize=(8, 6))
    uniq = sorted(np.unique(labels_arr).tolist())
    colors = plt.cm.tab10(np.linspace(0, 1, max(1, len(uniq))))
    for c, color in zip(uniq, colors):
        mask = labels_arr == c
        ax.scatter(scores[mask, 0], scores[mask, 1], s=26, alpha=0.8, color=color, label=f"C{int(c)}")
    ax.set_title(f"KMeans clusters: {block} | {dtag} | {k_label}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8, loc="best")
    fig.tight_layout()
    fig.savefig(out_path, dpi=180)
    plt.close(fig)


def run_pca_block_unsupervised(
    df: pd.DataFrame,
    outcome_col: str,
    block: str,
    numeric_cols: Sequence[str],
    out_paths: Dict[str, Path],
    max_components: int = 20,
) -> Dict[str, object]:
    X = build_numeric_matrix(df, numeric_cols)
    if X.empty:
        return {
            "block": block,
            "n_features": 0,
            "n_components": 0,
            "n_pcs_70": 0,
            "n_pcs_80": 0,
            "n_pcs_90": 0,
            "scores": np.empty((len(df), 0), dtype=float),
            "top_pc_features": {},
        }

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X.values)
    n_components = int(min(max_components, X_scaled.shape[1], X_scaled.shape[0]))
    if n_components < 1:
        return {
            "block": block,
            "n_features": int(X.shape[1]),
            "n_components": 0,
            "n_pcs_70": 0,
            "n_pcs_80": 0,
            "n_pcs_90": 0,
            "scores": np.empty((len(df), 0), dtype=float),
            "top_pc_features": {},
        }

    pca = PCA(n_components=n_components, random_state=SEED)
    scores = pca.fit_transform(X_scaled)
    evr = pca.explained_variance_ratio_
    cum = np.cumsum(evr)
    n_pcs_70 = compute_pc_threshold(cum, 0.70)
    n_pcs_80 = compute_pc_threshold(cum, 0.80)
    n_pcs_90 = compute_pc_threshold(cum, 0.90)

    variance_df = pd.DataFrame(
        {
            "block": block,
            "component": [f"PC{i}" for i in range(1, len(evr) + 1)],
            "explained_variance_ratio": evr,
            "cumulative_variance_ratio": cum,
        }
    )
    variance_df.to_csv(out_paths["tables"] / f"pca_variance_{block}.csv", index=False)

    loadings = pd.DataFrame(
        pca.components_.T,
        index=list(X.columns),
        columns=[f"PC{i}" for i in range(1, pca.components_.shape[0] + 1)],
    )
    max_pc_to_save = min(10, loadings.shape[1])
    top_rows: List[Dict[str, object]] = []
    top_pc_features: Dict[str, List[str]] = {}
    for i in range(1, max_pc_to_save + 1):
        pc = f"PC{i}"
        s = loadings[pc].abs().sort_values(ascending=False).head(20)
        top_pc_features[pc] = s.index.tolist()
        for rank, feat in enumerate(s.index.tolist(), start=1):
            top_rows.append(
                {
                    "block": block,
                    "pc": pc,
                    "rank": rank,
                    "feature": feat,
                    "loading_signed": float(loadings.loc[feat, pc]),
                    "loading_abs": float(abs(loadings.loc[feat, pc])),
                }
            )
    top_df = pd.DataFrame(top_rows)
    top_df.to_csv(out_paths["tables"] / f"pca_top_loadings_{block}_PC1-PC10.csv", index=False)

    x = np.arange(1, len(evr) + 1)
    fig, ax1 = plt.subplots(figsize=(9, 5))
    ax1.bar(x, evr, color="#4F81BD", alpha=0.85, label="Explained variance")
    ax1.plot(x, cum, color="#C0504D", marker="o", label="Cumulative variance")
    ax1.set_title(f"PCA Scree: {block}")
    ax1.set_xlabel("Principal Component")
    ax1.set_ylabel("Variance Ratio")
    ax1.set_xticks(x)
    ax1.grid(alpha=0.2)
    ax1.legend(loc="best", fontsize=8)
    fig.tight_layout()
    fig.savefig(out_paths["plots"] / f"pca_scree_{block}.png", dpi=180)
    plt.close(fig)

    score_cols = [f"PC{i}" for i in range(1, scores.shape[1] + 1)]
    score_df = pd.DataFrame(scores, columns=score_cols)
    score_df[outcome_col] = df[outcome_col].astype(str).values
    if {"PC1", "PC2"}.issubset(score_df.columns):
        plot_pca_scatter(
            score_df=score_df,
            x_col="PC1",
            y_col="PC2",
            outcome_col=outcome_col,
            out_file=out_paths["plots"] / f"pca_scatter_{block}_pc1_pc2_by_diagnosis.png",
            title=f"PCA {block}: PC1 vs PC2 by diagnosis",
        )
    if {"PC1", "PC3"}.issubset(score_df.columns):
        plot_pca_scatter(
            score_df=score_df,
            x_col="PC1",
            y_col="PC3",
            outcome_col=outcome_col,
            out_file=out_paths["plots"] / f"pca_scatter_{block}_pc1_pc3_by_diagnosis.png",
            title=f"PCA {block}: PC1 vs PC3 by diagnosis",
        )

    return {
        "block": block,
        "n_features": int(X.shape[1]),
        "n_components": int(scores.shape[1]),
        "n_pcs_70": int(n_pcs_70),
        "n_pcs_80": int(n_pcs_80),
        "n_pcs_90": int(n_pcs_90),
        "scores": scores,
        "top_pc_features": top_pc_features,
    }


def run_kmeans_for_block_dim(
    scores: np.ndarray,
    y_group: pd.Series,
    block: str,
    dtag: str,
    n_dim: int,
    out_paths: Dict[str, Path],
    random_state: int,
) -> Dict[str, object]:
    Xd = scores[:, :n_dim]
    if Xd.shape[1] < 2:
        return {"k_best": None, "silhouette_mean": np.nan}

    silhouette_rows: List[Dict[str, object]] = []
    k_to_labels: Dict[int, np.ndarray] = {}
    for k in range(2, 9):
        if k >= Xd.shape[0]:
            continue
        model = KMeans(n_clusters=k, n_init=50, random_state=random_state)
        labels = model.fit_predict(Xd)
        sil = silhouette_score(Xd, labels)
        k_to_labels[k] = labels
        silhouette_rows.append(
            {
                "block": block,
                "Dtag": dtag,
                "n_dim": int(n_dim),
                "k": int(k),
                "silhouette_score": float(sil),
            }
        )
    sil_df = pd.DataFrame(silhouette_rows).sort_values("k").reset_index(drop=True)
    sil_df.to_csv(out_paths["tables"] / f"kmeans_silhouette_{block}_{dtag}.csv", index=False)
    if sil_df.empty:
        return {"k_best": None, "silhouette_mean": np.nan}

    best_idx = sil_df["silhouette_score"].idxmax()
    k_best = int(sil_df.loc[best_idx, "k"])
    eval_targets: List[Tuple[str, int]] = [("k4", 4), ("k6", 6), ("kbest", k_best)]

    align_rows: List[Dict[str, object]] = []
    for k_label, k in eval_targets:
        if k not in k_to_labels:
            model = KMeans(n_clusters=k, n_init=50, random_state=random_state)
            labels = model.fit_predict(Xd)
            k_to_labels[k] = labels
        labels = k_to_labels[k]
        ari = adjusted_rand_score(y_group, labels)
        nmi = normalized_mutual_info_score(y_group, labels)
        align_rows.append(
            {
                "block": block,
                "Dtag": dtag,
                "n_dim": int(n_dim),
                "method": "kmeans",
                "k_label": k_label,
                "k": int(k),
                "ari": float(ari),
                "nmi": float(nmi),
            }
        )

        cont_file = out_paths["tables"] / f"kmeans_contingency_{block}_{dtag}_{k_label}.csv"
        save_contingency_with_percentages(labels, y_group.astype(str), cont_file)

        plot_file = out_paths["plots"] / f"kmeans_clusters_{block}_{dtag}_{k_label}.png"
        plot_kmeans_clusters_pc12(scores=Xd, cluster_labels=labels, block=block, dtag=dtag, k_label=k_label, out_path=plot_file)

    align_df = pd.DataFrame(align_rows)
    align_df.to_csv(out_paths["tables"] / f"clustering_alignment_{block}_{dtag}.csv", index=False)

    return {
        "k_best": k_best,
        "silhouette_mean": float(sil_df["silhouette_score"].mean()),
        "alignment_df": align_df,
    }


def run_hclust_for_block_dim(
    scores: np.ndarray,
    y_group: pd.Series,
    block: str,
    dtag: str,
    n_dim: int,
    out_paths: Dict[str, Path],
) -> Dict[str, float]:
    if linkage is None or dendrogram is None or fcluster is None:
        return {"k4_ari": np.nan, "k4_nmi": np.nan, "k6_ari": np.nan, "k6_nmi": np.nan}

    Xd = scores[:, :n_dim]
    if Xd.shape[1] < 2:
        return {"k4_ari": np.nan, "k4_nmi": np.nan, "k6_ari": np.nan, "k6_nmi": np.nan}

    Z = linkage(Xd, method="ward")
    fig, ax = plt.subplots(figsize=(10, 5))
    dendrogram(Z, truncate_mode="lastp", p=min(30, max(2, Xd.shape[0] - 1)), no_labels=True, ax=ax)
    ax.set_title(f"Ward Dendrogram (truncated): {block} | {dtag}")
    ax.set_xlabel("Cluster index (truncated)")
    ax.set_ylabel("Distance")
    fig.tight_layout()
    fig.savefig(out_paths["plots"] / f"hclust_dendrogram_{block}_{dtag}.png", dpi=180)
    plt.close(fig)

    metrics_out: Dict[str, float] = {}
    for k_label, k in [("k4", 4), ("k6", 6)]:
        labels = fcluster(Z, t=k, criterion="maxclust")
        cont_file = out_paths["tables"] / f"hclust_contingency_{block}_{dtag}_{k_label}.csv"
        save_contingency_with_percentages(labels, y_group.astype(str), cont_file)
        metrics_out[f"{k_label}_ari"] = float(adjusted_rand_score(y_group, labels))
        metrics_out[f"{k_label}_nmi"] = float(normalized_mutual_info_score(y_group, labels))
    return metrics_out


def run_unsupervised_structure(
    df: pd.DataFrame,
    outcome_col: str,
    col_groups: Dict[str, List[str]],
    args: argparse.Namespace,
    logger: logging.Logger,
) -> Path:
    out_paths = ensure_unsupervised_dirs(args.outdir)
    logger.info("Unsupervised output directory: %s", out_paths["root"])

    weather_before = len(col_groups["weather_model_numeric"])
    corr_out = {
        "root": out_paths["root"],
        "tables": out_paths["tables"],
        "plots": out_paths["plots"],
        "models": out_paths["root"],
    }
    kept_weather_numeric = correlation_filter_weather(
        df=df,
        weather_numeric_cols=col_groups["weather_model_numeric"],
        threshold=args.corr_threshold,
        out_paths=corr_out,
    )
    logger.info(
        "Unsupervised environmental numeric alignment (kept_weather_numeric): %d/%d",
        len(kept_weather_numeric),
        weather_before,
    )

    feature_blocks = build_feature_blocks(col_groups)
    env_numeric_original = set(col_groups["weather_model_numeric"])
    feature_blocks["environmental_only"]["numeric"] = sorted(kept_weather_numeric)
    feature_blocks["combined"]["numeric"] = sorted(
        [c for c in feature_blocks["combined"]["numeric"] if c not in env_numeric_original]
        + list(kept_weather_numeric)
    )

    block_numeric_map = {
        "environmental_only": list(feature_blocks["environmental_only"]["numeric"]),
        "clinical_only": list(feature_blocks["clinical_only"]["numeric"]),
        "combined": list(feature_blocks["combined"]["numeric"]),
    }
    rows = []
    for block, cols in block_numeric_map.items():
        for c in cols:
            rows.append({"block": block, "numeric_feature": c})
    pd.DataFrame(rows).to_csv(out_paths["tables"] / "unsupervised_block_numeric_features.csv", index=False)

    y_group = df[outcome_col].astype(str)
    pca_threshold_rows: List[Dict[str, object]] = []
    cluster_summary_rows: List[Dict[str, object]] = []
    block_top_pcs: Dict[str, Dict[str, List[str]]] = {}

    for block, numeric_cols in block_numeric_map.items():
        pca_meta = run_pca_block_unsupervised(
            df=df,
            outcome_col=outcome_col,
            block=block,
            numeric_cols=numeric_cols,
            out_paths=out_paths,
        )
        block_top_pcs[block] = pca_meta.get("top_pc_features", {})
        pca_threshold_rows.append(
            {
                "block": block,
                "n_features": int(pca_meta["n_features"]),
                "n_components": int(pca_meta["n_components"]),
                "n_pcs_70": int(pca_meta["n_pcs_70"]),
                "n_pcs_80": int(pca_meta["n_pcs_80"]),
                "n_pcs_90": int(pca_meta["n_pcs_90"]),
            }
        )
        if int(pca_meta["n_components"]) < 2:
            continue

        scores = np.asarray(pca_meta["scores"], dtype=float)
        d_fixed = min(6, scores.shape[1])
        d_fixed = max(2, d_fixed)
        d_80 = min(20, int(pca_meta["n_pcs_80"]))
        d_80 = max(2, d_80)
        dims = [("Dfixed6", d_fixed), ("D80", d_80)]

        for dtag, dval in dims:
            kmeans_meta = run_kmeans_for_block_dim(
                scores=scores,
                y_group=y_group,
                block=block,
                dtag=dtag,
                n_dim=dval,
                out_paths=out_paths,
                random_state=args.seed,
            )
            h_meta = run_hclust_for_block_dim(
                scores=scores,
                y_group=y_group,
                block=block,
                dtag=dtag,
                n_dim=dval,
                out_paths=out_paths,
            )
            align_df = kmeans_meta.get("alignment_df", pd.DataFrame())
            kbest_ari = np.nan
            kbest_nmi = np.nan
            if isinstance(align_df, pd.DataFrame) and not align_df.empty:
                sub = align_df[align_df["k_label"] == "kbest"]
                if not sub.empty:
                    kbest_ari = float(sub.iloc[0]["ari"])
                    kbest_nmi = float(sub.iloc[0]["nmi"])
            cluster_summary_rows.append(
                {
                    "block": block,
                    "Dtag": dtag,
                    "n_dim": int(dval),
                    "kmeans_k_best": kmeans_meta.get("k_best"),
                    "kmeans_kbest_ari": kbest_ari,
                    "kmeans_kbest_nmi": kbest_nmi,
                    "hclust_k4_ari": h_meta.get("k4_ari", np.nan),
                    "hclust_k4_nmi": h_meta.get("k4_nmi", np.nan),
                    "hclust_k6_ari": h_meta.get("k6_ari", np.nan),
                    "hclust_k6_nmi": h_meta.get("k6_nmi", np.nan),
                }
            )

    thresholds_df = pd.DataFrame(pca_threshold_rows)
    thresholds_df.to_csv(out_paths["tables"] / "pca_thresholds_summary.csv", index=False)
    cluster_summary_df = pd.DataFrame(cluster_summary_rows)
    cluster_summary_df.to_csv(out_paths["tables"] / "clustering_summary.csv", index=False)

    lines: List[str] = []
    lines.append("# Unsupervised Structure Summary")
    lines.append("")
    lines.append("## Scope")
    lines.append("- Numeric-only PCA and clustering aligned to modeling feature blocks.")
    lines.append("- Environmental PCA uses globally correlation-filtered `kept_weather_numeric` features (not raw weather numeric list).")
    lines.append("- This is exploratory structure analysis only; no causal interpretation.")
    lines.append("")
    lines.append("## PCA Thresholds")
    if thresholds_df.empty:
        lines.append("- No PCA outputs generated.")
    else:
        for _, r in thresholds_df.iterrows():
            lines.append(
                f"- {r['block']}: n_features={int(r['n_features'])}, n_pcs_70={int(r['n_pcs_70'])}, n_pcs_80={int(r['n_pcs_80'])}, n_pcs_90={int(r['n_pcs_90'])}."
            )
    lines.append("")
    lines.append("## PC1-PC3 Loading Signals (top contributors)")
    for block in ["environmental_only", "clinical_only", "combined"]:
        top_map = block_top_pcs.get(block, {})
        if not top_map:
            lines.append(f"- {block}: unavailable.")
            continue
        pc1 = ", ".join(top_map.get("PC1", [])[:5]) if top_map.get("PC1") else "n/a"
        pc2 = ", ".join(top_map.get("PC2", [])[:5]) if top_map.get("PC2") else "n/a"
        pc3 = ", ".join(top_map.get("PC3", [])[:5]) if top_map.get("PC3") else "n/a"
        lines.append(f"- {block}: PC1 [{pc1}]; PC2 [{pc2}]; PC3 [{pc3}].")
    lines.append("")
    lines.append("## Clustering Alignment to Diagnosis")
    if cluster_summary_df.empty:
        lines.append("- No clustering alignment outputs generated.")
    else:
        for block in ["environmental_only", "clinical_only", "combined"]:
            sub = cluster_summary_df[cluster_summary_df["block"] == block]
            if sub.empty:
                continue
            best_idx = sub["kmeans_kbest_nmi"].astype(float).idxmax()
            best = sub.loc[best_idx]
            lines.append(
                f"- {block}: best KMeans setting {best['Dtag']} (k_best={best['kmeans_k_best']}), ARI={best['kmeans_kbest_ari']:.3f}, NMI={best['kmeans_kbest_nmi']:.3f}."
            )
    lines.append("")
    lines.append("## Caveat")
    lines.append("- Clusters are not expected to perfectly reproduce clinician diagnosis labels; use as latent structure screening only.")
    lines.append("- Results are exploratory and non-causal.")
    if linkage is None:
        lines.append("- Ward hierarchical clustering was skipped because `scipy` is unavailable.")

    summary_path = out_paths["root"] / "unsupervised_summary.md"
    summary_path.write_text("\n".join(lines), encoding="utf-8")
    logger.info("Unsupervised summary: %s", summary_path)
    return summary_path


def correlation_filter_weather(
    df: pd.DataFrame, weather_numeric_cols: Sequence[str], threshold: float, out_paths: Dict[str, Path]
) -> List[str]:
    if not weather_numeric_cols:
        return []
    X = df[list(weather_numeric_cols)].copy()
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.median(numeric_only=True))

    corr = X.corr(numeric_only=True).abs()
    corr.to_csv(out_paths["tables"] / "weather_correlation_matrix_abs.csv", index=True)

    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if (upper[col] > threshold).any()]
    kept = [c for c in weather_numeric_cols if c not in to_drop]

    summary = pd.DataFrame(
        {
            "metric": [
                "n_weather_numeric_before",
                "n_dropped_high_corr",
                "n_weather_numeric_after",
                "corr_threshold",
            ],
            "value": [len(weather_numeric_cols), len(to_drop), len(kept), threshold],
        }
    )
    summary.to_csv(out_paths["tables"] / "weather_correlation_filter_summary.csv", index=False)
    pd.DataFrame({"dropped_feature": to_drop}).to_csv(
        out_paths["tables"] / "weather_features_dropped_by_correlation.csv", index=False
    )
    pd.DataFrame({"kept_feature": kept}).to_csv(
        out_paths["tables"] / "weather_features_kept_after_correlation.csv", index=False
    )


    if len(corr.columns) > 0:
        vari = X.var().sort_values(ascending=False)
        view_cols = vari.head(min(45, len(vari))).index.tolist()
        corr_view = corr.loc[view_cols, view_cols]
        fig, ax = plt.subplots(figsize=(12, 10))
        im = ax.imshow(corr_view.values, cmap="coolwarm", vmin=0, vmax=1)
        ax.set_title("Weather Correlation Matrix (Abs) - Top Variance Features")
        ax.set_xticks(range(len(view_cols)))
        ax.set_yticks(range(len(view_cols)))
        ax.set_xticklabels(view_cols, rotation=90, fontsize=7)
        ax.set_yticklabels(view_cols, fontsize=7)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        fig.tight_layout()
        fig.savefig(out_paths["plots"] / "weather_correlation_heatmap_abs_topvariance.png", dpi=180)
        plt.close(fig)

    return kept


def filter_correlated_numeric(
    df: pd.DataFrame,
    numeric_cols: Sequence[str],
    threshold: float,
) -> List[str]:
    if not numeric_cols:
        return []
    X = df[list(numeric_cols)].copy()
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.median(numeric_only=True))
    corr = X.corr(numeric_only=True).abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if (upper[col] > threshold).any()]
    kept = [c for c in numeric_cols if c not in to_drop]
    return kept


def run_logistic_odds_ratio(
    df: pd.DataFrame,
    outcome_col: str,
    features: Sequence[str],
    out_paths: Dict[str, Path],
    random_state: int,
    bootstrap_repeats: int = 250,
    output_csv_name: str = "logistic_odds_ratios_weather.csv",
    output_plot_name: str = "logistic_odds_ratio_top_features.png",
    write_outputs: bool = True,
) -> pd.DataFrame:
    if not features:
        return pd.DataFrame()

    X_raw = df[list(features)].copy()
    X_raw = X_raw.apply(pd.to_numeric, errors="coerce")
    X_raw = X_raw.fillna(X_raw.median(numeric_only=True))
    y = df[outcome_col].astype(str).values
    classes = sorted(np.unique(y).tolist())

    scaler = StandardScaler()
    X = scaler.fit_transform(X_raw.values)

    rng = np.random.default_rng(random_state)
    rows: List[Dict[str, object]] = []

    for cls in classes:
        y_bin = (y == cls).astype(int)
        if y_bin.sum() < 10:
            continue

        base_model = LogisticRegression(
            solver="liblinear",
            class_weight="balanced",
            max_iter=3000,
            random_state=random_state,
        )
        base_model.fit(X, y_bin)
        coef = base_model.coef_[0]

        boot = []
        n = len(y_bin)
        for _ in range(bootstrap_repeats):
            idx = rng.integers(0, n, size=n)
            y_s = y_bin[idx]
            if y_s.min() == y_s.max():
                continue
            m = LogisticRegression(
                solver="liblinear",
                class_weight="balanced",
                max_iter=2000,
                random_state=random_state,
            )
            try:
                m.fit(X[idx], y_s)
            except Exception:
                continue
            boot.append(m.coef_[0])

        boot_arr = np.array(boot) if boot else np.empty((0, len(features)))

        for j, feat in enumerate(features):
            or_val = float(np.exp(coef[j]))
            ci_low = float(np.exp(np.percentile(boot_arr[:, j], 2.5))) if boot_arr.shape[0] > 20 else np.nan
            ci_high = float(np.exp(np.percentile(boot_arr[:, j], 97.5))) if boot_arr.shape[0] > 20 else np.nan
            rows.append(
                {
                    "outcome_class": cls,
                    "feature": feat,
                    "coef": float(coef[j]),
                    "odds_ratio": or_val,
                    "ci95_low": ci_low,
                    "ci95_high": ci_high,
                    "direction": "positive" if coef[j] > 0 else "negative",
                }
            )

    if not rows:
        return pd.DataFrame()

    or_df = pd.DataFrame(rows)
    or_df["abs_log_or"] = np.abs(np.log(or_df["odds_ratio"].clip(lower=1e-8)))
    or_df = or_df.sort_values(["outcome_class", "abs_log_or"], ascending=[True, False]).reset_index(drop=True)
    if write_outputs:
        or_df.to_csv(out_paths["tables"] / output_csv_name, index=False)
        plot_or_summary(or_df, out_paths["plots"] / output_plot_name)
    return or_df


def plot_or_summary(or_df: pd.DataFrame, out_file: Path) -> None:
    classes = sorted(or_df["outcome_class"].unique().tolist())
    nrows = len(classes)
    fig, axes = plt.subplots(nrows=nrows, ncols=1, figsize=(11, max(4, 3.8 * nrows)))
    if nrows == 1:
        axes = [axes]

    for ax, cls in zip(axes, classes):
        sub = or_df[or_df["outcome_class"] == cls].head(10).copy()
        sub = sub.iloc[::-1]
        ax.barh(sub["feature"], sub["odds_ratio"], color="#3A7D44")
        ax.axvline(1.0, color="black", linestyle="--", linewidth=1)
        ax.set_title(f"Top OR Features: {cls}")
        ax.set_xlabel("Odds Ratio (standardized predictors)")
        ax.set_ylabel("Feature")
    fig.tight_layout()
    fig.savefig(out_file, dpi=180)
    plt.close(fig)


def run_moon_encoding_sensitivity(
    df: pd.DataFrame,
    outcome_col: str,
    col_groups: Dict[str, List[str]],
    corr_threshold: float,
    out_paths: Dict[str, Path],
    random_state: int,
    test_size: float,
    bootstrap_repeats: int,
) -> pd.DataFrame:
    base_numeric = col_groups["weather_model_numeric_base"]
    base_categorical = col_groups["weather_model_categorical_base"]

    scenarios = [
        {
            "encoding": "sincos_fraction",
            "moon_numeric": col_groups["moon_default_numeric"],
            "moon_categorical": [],
        },
        {
            "encoding": "thresholded",
            "moon_numeric": col_groups["moon_threshold_numeric"],
            "moon_categorical": col_groups["moon_threshold_categorical"],
        },
        {
            "encoding": "mixed",
            "moon_numeric": col_groups["moon_mixed_numeric"],
            "moon_categorical": col_groups["moon_mixed_categorical"],
        },
    ]

    rows: List[Dict[str, object]] = []
    for sc in scenarios:
        encoding = str(sc["encoding"])
        model_numeric = sorted(set(base_numeric + list(sc["moon_numeric"])))
        model_categorical = sorted(set(base_categorical + list(sc["moon_categorical"])))
        kept_numeric = filter_correlated_numeric(df, model_numeric, corr_threshold)

        run_name = f"rf_environmental_moon_{encoding}"
        rf_metrics, rf_importance = run_random_forest(
            df=df,
            outcome_col=outcome_col,
            numeric_cols=kept_numeric,
            categorical_cols=model_categorical,
            out_paths=out_paths,
            run_name=run_name,
            random_state=random_state,
            test_size=test_size,
        )

        top_num = (
            rf_importance[rf_importance["feature"].str.startswith("num__", na=False)]
            .head(12)["feature"]
            .str.replace("num__", "", regex=False)
            .tolist()
        )
        or_df = run_logistic_odds_ratio(
            df=df,
            outcome_col=outcome_col,
            features=top_num,
            out_paths=out_paths,
            random_state=random_state,
            bootstrap_repeats=bootstrap_repeats,
            output_csv_name=f"logistic_odds_ratios_{encoding}.csv",
            output_plot_name=f"logistic_odds_ratio_{encoding}.png",
            write_outputs=False,
        )
        top_or_feature = ""
        top_or_value = np.nan
        if not or_df.empty:
            idx = or_df["abs_log_or"].idxmax()
            top_or_feature = str(or_df.loc[idx, "feature"])
            top_or_value = float(or_df.loc[idx, "odds_ratio"])

        rows.append(
            {
                "encoding": encoding,
                "moon_numeric_used": ", ".join(sc["moon_numeric"]) if sc["moon_numeric"] else "(none)",
                "moon_categorical_used": ", ".join(sc["moon_categorical"]) if sc["moon_categorical"] else "(none)",
                "numeric_features_before_corr": len(model_numeric),
                "numeric_features_after_corr": len(kept_numeric),
                "categorical_features_used": len(model_categorical),
                "rf_accuracy": float(rf_metrics["accuracy"]),
                "rf_balanced_accuracy": float(rf_metrics["balanced_accuracy"]),
                "rf_macro_f1": float(rf_metrics["macro_f1"]),
                "logistic_top_or_feature": top_or_feature,
                "logistic_top_or_value": top_or_value,
                "logistic_n_rows": int(len(or_df)),
            }
        )

    out_df = pd.DataFrame(rows).sort_values("rf_balanced_accuracy", ascending=False).reset_index(drop=True)
    out_df.to_csv(out_paths["tables"] / "moon_encoding_sensitivity.csv", index=False)
    return out_df


def save_classic_eda_block(
    df: pd.DataFrame,
    date_col: str | None,
    weather_numeric_cols: Sequence[str],
    moon_numeric_cols: Sequence[str],
    moon_categorical_cols: Sequence[str],
    out_paths: Dict[str, Path],
) -> Dict[str, int]:
    summary: Dict[str, int] = {}


    missing_n = df.isna().sum()
    missing_pct = (100.0 * missing_n / len(df)).round(3)
    miss_df = pd.DataFrame({"variable": df.columns, "missing_n": missing_n.values, "missing_pct": missing_pct.values})
    miss_df = miss_df.sort_values("missing_pct", ascending=False).reset_index(drop=True)
    miss_df.to_csv(out_paths["tables"] / "missingness_per_column.csv", index=False)
    summary["missing_variables"] = int((miss_df["missing_n"] > 0).sum())

    top30 = miss_df.head(30).copy().iloc[::-1]
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(top30["variable"], top30["missing_pct"], color="#4C78A8")
    ax.set_title("Top 30 Missingness Percent")
    ax.set_xlabel("Missing %")
    ax.set_ylabel("Variable")
    fig.tight_layout()
    fig.savefig(out_paths["plots"] / "missingness_top30_bar.png", dpi=180)
    plt.close(fig)


    heat_cols = miss_df.head(min(60, len(miss_df)))["variable"].tolist()
    miss_mat = df[heat_cols].isna().astype(int).values
    fig, ax = plt.subplots(figsize=(12, 8))
    im = ax.imshow(miss_mat, aspect="auto", cmap="Greys")
    ax.set_title("Missingness Heatmap (Top Missingness Columns)")
    ax.set_xlabel("Variables")
    ax.set_ylabel("Rows")
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(out_paths["plots"] / "missingness_heatmap.png", dpi=180)
    plt.close(fig)


    dt = pd.to_datetime(df[date_col], errors="coerce") if date_col else pd.Series([pd.NaT] * len(df))
    if dt.notna().sum() > 0:
        month_name = dt.dt.month.map(
            {
                1: "Jan",
                2: "Feb",
                3: "Mar",
                4: "Apr",
                5: "May",
                6: "Jun",
                7: "Jul",
                8: "Aug",
                9: "Sep",
                10: "Oct",
                11: "Nov",
                12: "Dec",
            }
        )
        month_count = month_name.value_counts().reindex(
            ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"],
            fill_value=0,
        )
        month_count_df = month_count.rename_axis("month").reset_index(name="case_count")
        month_count_df.to_csv(out_paths["tables"] / "case_count_by_month.csv", index=False)

        fig, ax = plt.subplots(figsize=(9, 4))
        ax.bar(month_count_df["month"], month_count_df["case_count"], color="#2A9D8F")
        ax.set_title("Case Count by Month")
        ax.set_xlabel("Month")
        ax.set_ylabel("Count")
        fig.tight_layout()
        fig.savefig(out_paths["plots"] / "case_count_by_month.png", dpi=180)
        plt.close(fig)


        trend_candidates = [
            "temperature_2m_mean__day1",
            "shortwave_radiation_sum__day1",
            "uv_index_max__day1",
            "wind_speed_10m_max__day1",
            "precipitation_sum__day1",
        ]
        trend_cols = [c for c in trend_candidates if c in df.columns]
        if trend_cols:
            trend_df = df.copy()
            trend_df["__month"] = month_name
            monthly = trend_df.groupby("__month")[trend_cols].mean(numeric_only=True)
            monthly = monthly.reindex(
                ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
            )
            monthly.to_csv(out_paths["tables"] / "monthly_mean_trends.csv", index=True)

            fig, ax = plt.subplots(figsize=(10, 5))
            for c in trend_cols:
                ax.plot(monthly.index, monthly[c], marker="o", label=c)
            ax.set_title("Monthly Mean Trends (Key Weather/UV Features)")
            ax.set_xlabel("Month")
            ax.set_ylabel("Mean value")
            ax.legend(fontsize=7, loc="best")
            ax.grid(alpha=0.2)
            fig.tight_layout()
            fig.savefig(out_paths["plots"] / "monthly_mean_trends.png", dpi=180)
            plt.close(fig)


        if "shortwave_radiation_sum__day1" in df.columns:
            tmp = pd.DataFrame({"month": month_name, "value": pd.to_numeric(df["shortwave_radiation_sum__day1"], errors="coerce")})
            boxplot_by_month(tmp, "Shortwave Radiation Day1 by Month", out_paths["plots"] / "shortwave_by_month_boxplot.png")
        if "temperature_2m_mean__day1" in df.columns:
            tmp = pd.DataFrame({"month": month_name, "value": pd.to_numeric(df["temperature_2m_mean__day1"], errors="coerce")})
            boxplot_by_month(tmp, "Temperature Mean Day1 by Month", out_paths["plots"] / "temp_mean_by_month_boxplot.png")
        if "uv_index_max__day1" in df.columns:
            tmp = pd.DataFrame({"month": month_name, "value": pd.to_numeric(df["uv_index_max__day1"], errors="coerce")})
            boxplot_by_month(tmp, "UV Index Day1 by Month", out_paths["plots"] / "uv_day1_by_month_boxplot.png")


    key_cols = [
        c
        for c in [
            "temperature_2m_mean__day1",
            "temperature_2m_max__day1",
            "temperature_2m_min__day1",
            "precipitation_sum__day1",
            "rain_sum__day1",
            "shortwave_radiation_sum__day1",
            "sunshine_duration__day1",
            "daylight_duration__day1",
            "wind_speed_10m_max__day1",
            "wind_gusts_10m_max__day1",
            "surface_pressure_max__day1",
            "uv_index_max__day1",
        ]
        if c in df.columns
    ]
    if key_cols:
        plot_hist_grid(df, key_cols, out_paths["plots"] / "weather_key_histograms.png", "Key Weather Feature Histograms")


    if weather_numeric_cols:
        wn = df[list(weather_numeric_cols)].apply(pd.to_numeric, errors="coerce")
        wn = wn.fillna(wn.median(numeric_only=True))
        corr = wn.corr(numeric_only=True)
        view_cols = wn.var().sort_values(ascending=False).head(min(40, wn.shape[1])).index.tolist()
        corr_view = corr.loc[view_cols, view_cols]
        fig, ax = plt.subplots(figsize=(11, 10))
        im = ax.imshow(corr_view.values, vmin=-1, vmax=1, cmap="coolwarm")
        ax.set_title("Weather Correlation Heatmap")
        ax.set_xticks(range(len(view_cols)))
        ax.set_yticks(range(len(view_cols)))
        ax.set_xticklabels(view_cols, rotation=90, fontsize=7)
        ax.set_yticklabels(view_cols, fontsize=7)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        fig.tight_layout()
        fig.savefig(out_paths["plots"] / "weather_correlation_heatmap.png", dpi=180)
        plt.close(fig)


    uv_cols = [c for c in df.columns if "uv_" in c.lower()]
    summary["uv_columns"] = len(uv_cols)
    if uv_cols:
        plot_hist_grid(df, uv_cols[:12], out_paths["plots"] / "uv_histograms.png", "UV Feature Histograms")

        uv_num = df[uv_cols].apply(pd.to_numeric, errors="coerce")
        uv_num = uv_num.fillna(uv_num.median(numeric_only=True))

        fig, ax = plt.subplots(figsize=(10, 6))
        ax.boxplot([uv_num[c].dropna().values for c in uv_cols[:12]], tick_labels=uv_cols[:12], vert=False)
        ax.set_title("UV Feature Boxplot")
        ax.set_xlabel("Value")
        fig.tight_layout()
        fig.savefig(out_paths["plots"] / "uv_feature_boxplot.png", dpi=180)
        plt.close(fig)


        uv_corr = uv_num.corr(numeric_only=True)
        fig, ax = plt.subplots(figsize=(9, 8))
        im = ax.imshow(uv_corr.values, vmin=-1, vmax=1, cmap="coolwarm")
        ax.set_title("UV Correlation Heatmap")
        ax.set_xticks(range(len(uv_corr.columns)))
        ax.set_yticks(range(len(uv_corr.columns)))
        ax.set_xticklabels(uv_corr.columns, rotation=90, fontsize=7)
        ax.set_yticklabels(uv_corr.columns, fontsize=7)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        fig.tight_layout()
        fig.savefig(out_paths["plots"] / "uv_correlation_heatmap.png", dpi=180)
        plt.close(fig)


        anchor_weather = [c for c in ["temperature_2m_mean__day1", "shortwave_radiation_sum__day1", "sunshine_duration__day1", "daylight_duration__day1"] if c in df.columns]
        if anchor_weather:
            mix = pd.concat([uv_num, df[anchor_weather].apply(pd.to_numeric, errors="coerce")], axis=1)
            mix = mix.fillna(mix.median(numeric_only=True))
            mix_corr = mix.corr(numeric_only=True).loc[uv_cols, uv_cols + anchor_weather]
            fig, ax = plt.subplots(figsize=(10, max(4, 0.4 * len(uv_cols) + 1)))
            im = ax.imshow(mix_corr.values, vmin=-1, vmax=1, cmap="coolwarm", aspect="auto")
            ax.set_title("UV vs Weather Correlation Heatmap")
            ax.set_xticks(range(len(mix_corr.columns)))
            ax.set_yticks(range(len(mix_corr.index)))
            ax.set_xticklabels(mix_corr.columns, rotation=90, fontsize=8)
            ax.set_yticklabels(mix_corr.index, fontsize=8)
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            fig.tight_layout()
            fig.savefig(out_paths["plots"] / "uv_weather_correlation_heatmap.png", dpi=180)
            plt.close(fig)

        if {"uv_index_max__day1", "shortwave_radiation_sum__day1"}.issubset(df.columns):
            fig, ax = plt.subplots(figsize=(7, 5))
            ax.scatter(
                pd.to_numeric(df["shortwave_radiation_sum__day1"], errors="coerce"),
                pd.to_numeric(df["uv_index_max__day1"], errors="coerce"),
                s=18,
                alpha=0.65,
                color="#33658A",
            )
            ax.set_title("UV Day1 vs Shortwave Radiation Day1")
            ax.set_xlabel("shortwave_radiation_sum__day1")
            ax.set_ylabel("uv_index_max__day1")
            ax.grid(alpha=0.2)
            fig.tight_layout()
            fig.savefig(out_paths["plots"] / "uv_vs_shortwave_scatter.png", dpi=180)
            plt.close(fig)


        if date_col and "uv_index_max__day1" in df.columns:
            dt2 = pd.to_datetime(df[date_col], errors="coerce")
            tmp = pd.DataFrame(
                {
                    "month": dt2.dt.month,
                    "uv_day1": pd.to_numeric(df["uv_index_max__day1"], errors="coerce"),
                }
            )
            tmp = tmp.dropna()
            if not tmp.empty:
                grp = tmp.groupby("month")["uv_day1"]
                z = (tmp["uv_day1"] - grp.transform("mean")) / grp.transform("std").replace(0, np.nan)
                z = z.replace([np.inf, -np.inf], np.nan).dropna()
                if len(z) > 5:
                    fig, ax = plt.subplots(figsize=(7, 5))
                    plot_hist_with_density(
                        ax=ax,
                        values=z.values,
                        bins=24,
                        hist_color="#7A9E9F",
                        density_color="#C1121F",
                    )
                    ax.set_title("UV Day1 Monthly Z-score Histogram")
                    ax.set_xlabel("z-score (within month)")
                    ax.set_ylabel("Count")
                    fig.tight_layout()
                    fig.savefig(out_paths["plots"] / "uv_day1_monthly_z_hist.png", dpi=180)
                    plt.close(fig)


    summary["moon_numeric_columns"] = len(moon_numeric_cols)
    summary["moon_categorical_columns"] = len(moon_categorical_cols)
    for col in moon_numeric_cols:
        vals = pd.to_numeric(df[col], errors="coerce").dropna()
        if len(vals) < 2:
            continue
        fig, ax = plt.subplots(figsize=(7, 4))
        plot_hist_with_density(
            ax=ax,
            values=vals.values,
            bins=24,
            hist_color="#6C757D",
            density_color="#C1121F",
        )
        ax.set_title(f"Moon Histogram: {col}")
        ax.set_xlabel(col)
        ax.set_ylabel("Count")
        fig.tight_layout()
        fig.savefig(out_paths["plots"] / f"moon_{col}_hist.png", dpi=180)
        plt.close(fig)

    for col in moon_categorical_cols:
        vc = df[col].astype(str).fillna("missing").value_counts()
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(vc.index.astype(str), vc.values, color="#355070")
        ax.set_title(f"Moon Category Bar: {col}")
        ax.set_xlabel(col)
        ax.set_ylabel("Count")
        ax.tick_params(axis="x", labelrotation=45)
        fig.tight_layout()
        if "moon_phase_category" in col:
            fig.savefig(out_paths["plots"] / "moon_phase_category_bar.png", dpi=180)
        else:
            fig.savefig(out_paths["plots"] / f"moon_{col}_bar.png", dpi=180)
        plt.close(fig)

    return summary


def boxplot_by_month(df_month_value: pd.DataFrame, title: str, out_file: Path) -> None:
    order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    grouped = []
    labels = []
    for m in order:
        vals = df_month_value.loc[df_month_value["month"] == m, "value"].dropna().values
        if len(vals) == 0:
            continue
        grouped.append(vals)
        labels.append(m)
    if not grouped:
        return
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.boxplot(grouped, tick_labels=labels, showfliers=False)
    ax.set_title(title)
    ax.set_xlabel("Month")
    ax.set_ylabel("Value")
    fig.tight_layout()
    fig.savefig(out_file, dpi=180)
    plt.close(fig)


def plot_hist_with_density(
    ax: plt.Axes,
    values: Sequence[float],
    bins: int = 24,
    hist_color: str = "#669BBC",
    density_color: str = "#C1121F",
) -> None:
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return

    counts, bin_edges, _ = ax.hist(
        vals,
        bins=bins,
        color=hist_color,
        edgecolor="white",
        alpha=0.82,
    )


    if vals.size < 5:
        return
    std = float(np.std(vals, ddof=1))
    if not np.isfinite(std) or std <= 0:
        return
    iqr = float(np.subtract(*np.percentile(vals, [75, 25])))
    sigma = min(std, iqr / 1.34) if iqr > 0 else std
    bw = 0.9 * sigma * (vals.size ** (-1.0 / 5.0))
    if not np.isfinite(bw) or bw <= 0:
        return

    x = np.linspace(float(vals.min()), float(vals.max()), 256)
    z = (x[:, None] - vals[None, :]) / bw
    density = np.exp(-0.5 * (z ** 2)).sum(axis=1) / (vals.size * bw * np.sqrt(2.0 * np.pi))


    bin_width = float(np.mean(np.diff(bin_edges))) if len(bin_edges) > 1 else 1.0
    y = density * vals.size * bin_width
    if np.all(np.isfinite(y)):
        ax.plot(x, y, color=density_color, linewidth=2.0)


def plot_hist_grid(df: pd.DataFrame, columns: Sequence[str], out_file: Path, title: str) -> None:
    cols = list(columns)
    if not cols:
        return
    n = len(cols)
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(12, 3.2 * nrows))
    axes = np.array(axes).reshape(-1)
    for i, c in enumerate(cols):
        ax = axes[i]
        vals = pd.to_numeric(df[c], errors="coerce").dropna()
        plot_hist_with_density(
            ax=ax,
            values=vals.values,
            bins=24,
            hist_color="#669BBC",
            density_color="#C1121F",
        )
        ax.set_title(c, fontsize=8)
        ax.tick_params(labelsize=7)
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(out_file, dpi=180)
    plt.close(fig)


def build_markdown_report(
    out_paths: Dict[str, Path],
    dataset_shape: Tuple[int, int],
    outcome_col: str,
    outcome_dist: pd.DataFrame,
    feature_group_summary: pd.DataFrame,
    feature_blocks_summary: pd.DataFrame,
    cv_summary_df: pd.DataFrame,
    pca_evr_df: pd.DataFrame,
    rf_all_metrics: Dict[str, float],
    rf_weather_metrics: Dict[str, float],
    corr_threshold: float,
    weather_before: int,
    weather_after: int,
    logistic_or_df: pd.DataFrame,
    classic_summary: Dict[str, int],
    moon_sensitivity_df: pd.DataFrame,
) -> str:
    lines: List[str] = []
    lines.append("# EDA v3: Weather-Focused Reproduction (HTML-style)")
    lines.append("")
    lines.append("This run follows the structure of the prior weather-results workflow (PCA -> filtering -> RF -> OR) on the enriched feline dataset.")
    lines.append("")
    lines.append("## Dataset Snapshot")
    lines.append(f"- Shape: **{dataset_shape[0]} rows x {dataset_shape[1]} columns**")
    lines.append(f"- Outcome column: **`{outcome_col}`**")
    lines.append("")
    lines.append("### Outcome Distribution")
    lines.append(df_to_markdown_table(outcome_dist, ["outcome", "count", "pct"]))
    lines.append("")
    lines.append("### Feature Grouping Summary")
    lines.append(df_to_markdown_table(feature_group_summary, ["group", "n_columns"]))
    lines.append("")
    lines.append("### Reusable Feature Blocks")
    lines.append(df_to_markdown_table(feature_blocks_summary, ["feature_block", "n_numeric", "n_categorical", "n_total"]))
    lines.append("")
    lines.append("## Stratified 5-Fold CV (Primary Evaluation)")
    if cv_summary_df.empty:
        lines.append("- CV summary unavailable.")
    else:
        lines.append(df_to_markdown_table(
            cv_summary_df,
            [
                "model",
                "feature_block",
                "accuracy_mean_pm_std",
                "balanced_accuracy_mean_pm_std",
                "macro_f1_mean_pm_std",
                "weighted_f1_mean_pm_std",
            ],
        ))
    lines.append("")
    lines.append("## Legacy EDA Block (Carried into v3)")
    lines.append("- Existing EDA style outputs are included in this run (missingness, monthly trends, weather/UV/moon plots).")
    lines.append(f"- UV columns profiled: **{classic_summary.get('uv_columns', 0)}**")
    lines.append(f"- Moon numeric columns profiled: **{classic_summary.get('moon_numeric_columns', 0)}**")
    lines.append(f"- Moon categorical columns profiled: **{classic_summary.get('moon_categorical_columns', 0)}**")
    lines.append(f"- Variables with any missingness: **{classic_summary.get('missing_variables', 0)}**")
    lines.append("")
    lines.append("## PCA Analysis (Weather Block)")
    if pca_evr_df.empty:
        lines.append("- PCA skipped (no weather numeric features detected).")
    else:
        pc1 = float(pca_evr_df.loc[pca_evr_df["component"] == "PC1", "explained_variance_ratio"].iloc[0]) if "PC1" in pca_evr_df["component"].values else np.nan
        pc2 = float(pca_evr_df.loc[pca_evr_df["component"] == "PC2", "explained_variance_ratio"].iloc[0]) if "PC2" in pca_evr_df["component"].values else np.nan
        pc3 = float(pca_evr_df.loc[pca_evr_df["component"] == "PC3", "explained_variance_ratio"].iloc[0]) if "PC3" in pca_evr_df["component"].values else np.nan
        lines.append(f"- PC1 variance ratio: **{pc1:.4f}**")
        lines.append(f"- PC2 variance ratio: **{pc2:.4f}**")
        lines.append(f"- PC3 variance ratio: **{pc3:.4f}**")
        lines.append("- See scree and loading outputs for interpretation.")
    lines.append("")
    lines.append("## RF Model (All Usable Features)")
    lines.append(f"- Accuracy: **{rf_all_metrics.get('accuracy', np.nan):.4f}**")
    lines.append(f"- Balanced accuracy: **{rf_all_metrics.get('balanced_accuracy', np.nan):.4f}**")
    lines.append(f"- Macro F1: **{rf_all_metrics.get('macro_f1', np.nan):.4f}**")
    lines.append("")
    lines.append("## Filtering Weather Parameters (Correlation Matrix)")
    lines.append(f"- Correlation threshold: **|r| > {corr_threshold:.2f}**")
    lines.append(f"- Weather numeric before: **{weather_before}**")
    lines.append(f"- Weather numeric after: **{weather_after}**")
    lines.append("")
    lines.append("## RF Model (Environmental Conditions)")
    lines.append(f"- Accuracy: **{rf_weather_metrics.get('accuracy', np.nan):.4f}**")
    lines.append(f"- Balanced accuracy: **{rf_weather_metrics.get('balanced_accuracy', np.nan):.4f}**")
    lines.append(f"- Macro F1: **{rf_weather_metrics.get('macro_f1', np.nan):.4f}**")
    lines.append("")
    lines.append("## Logistic Model (Odds Ratios)")
    if logistic_or_df.empty:
        lines.append("- Logistic OR section skipped (insufficient selected predictors).")
    else:
        top = logistic_or_df.groupby("outcome_class", as_index=False).first()[["outcome_class", "feature", "odds_ratio"]]
        lines.append("- Top OR feature per class:")
        lines.append(df_to_markdown_table(top, ["outcome_class", "feature", "odds_ratio"]))
    lines.append("")
    lines.append("## Moon Encoding Sensitivity")
    if moon_sensitivity_df.empty:
        lines.append("- Sensitivity block not available.")
    else:
        lines.append(
            "- RF + logistic screening rerun under three moon encodings: "
            "sincos(+fraction), thresholded flags(+category), and mixed."
        )
        lines.append(
            df_to_markdown_table(
                moon_sensitivity_df,
                [
                    "encoding",
                    "rf_accuracy",
                    "rf_balanced_accuracy",
                    "rf_macro_f1",
                    "numeric_features_after_corr",
                    "logistic_top_or_feature",
                    "logistic_top_or_value",
                ],
            )
        )
    lines.append("")
    lines.append("## Interpretation Notes")
    lines.append("- Exploratory only; no causal inference.")
    lines.append("- Correlation filtering reduces redundancy but can remove potentially useful interactions.")
    lines.append("- RF importances are relative; OR effects are class-specific and model-dependent.")
    lines.append("")
    lines.append("## Output Paths")
    lines.append(f"- Root: `{out_paths['root']}`")
    lines.append(f"- Tables: `{out_paths['tables']}`")
    lines.append(f"- Plots: `{out_paths['plots']}`")
    lines.append(f"- Models: `{out_paths['models']}`")
    return "\n".join(lines)


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="EDA v3 weather-focused run.")
    parser.add_argument("--input", type=Path, default=DEFAULT_INPUT, help="Path to enriched dataset (.csv or .xlsx)")
    parser.add_argument("--outdir", type=Path, default=DEFAULT_OUTDIR, help="Output directory")
    parser.add_argument("--corr-threshold", type=float, default=0.85, help="Absolute correlation threshold for filtering weather numeric features")
    parser.add_argument("--test-size", type=float, default=0.2, help="Test split size for RF models")
    parser.add_argument("--seed", type=int, default=SEED, help="Random seed")
    parser.add_argument("--or-bootstrap", type=int, default=250, help="Bootstrap repeats for OR confidence intervals")
    parser.add_argument("--cv-folds", type=int, default=5, help="Number of stratified CV folds for RF and logistic OvR")
    parser.add_argument(
        "--run-shap-stability",
        dest="run_shap_stability",
        action="store_true",
        default=(shap is not None),
        help="Run SHAP stability metrics/plots for binary HistGB (default: true when SHAP is installed).",
    )
    parser.add_argument(
        "--no-shap-stability",
        dest="run_shap_stability",
        action="store_false",
        help="Disable SHAP stability metrics/plots.",
    )
    parser.add_argument(
        "--run-permutation-test",
        action="store_true",
        default=False,
        help="Run environmental-only permutation test for binary HistGB ROC-AUC.",
    )
    parser.add_argument(
        "--perm-n",
        type=int,
        default=200,
        help="Number of permutations for environmental-only binary HistGB permutation test.",
    )
    parser.add_argument(
        "--perm-jobs",
        type=int,
        default=None,
        help="Number of parallel jobs for permutation indices (joblib loky). If omitted, auto-select based on CPU/load.",
    )
    parser.add_argument(
        "--run-lightgbm-replication",
        action="store_true",
        default=False,
        help="Run LightGBM replication path for binary CV + env-only permutation outputs.",
    )
    parser.add_argument(
        "--run-unsupervised",
        action="store_true",
        default=False,
        help="Run only the model-aligned unsupervised PCA + clustering module (outputs/eda_v3/unsupervised).",
    )
    return parser.parse_args()


def detect_runtime_parallelism() -> Tuple[int, float]:
    cpu_count = os.cpu_count() or 1
    load_avg = os.getloadavg()[0]
    return int(cpu_count), float(load_avg)


def choose_perm_jobs(cpu_count: int, load_avg: float, user_perm_jobs: int | None) -> Tuple[int, str]:
    if user_perm_jobs is not None:
        return max(1, int(user_perm_jobs)), "manual"
    if load_avg < (cpu_count * 0.5):
        return max(1, cpu_count - 4), "auto_low_load"
    return max(1, int(cpu_count / 2)), "auto_high_load"


def load_dataset(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Input not found: {path}")
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported format: {suffix}")


def main() -> None:
    args = parse_args()
    logger = setup_logging()
    np.random.seed(args.seed)
    cpu_count, load_avg = detect_runtime_parallelism()
    perm_jobs, perm_jobs_mode = choose_perm_jobs(cpu_count, load_avg, args.perm_jobs)
    logger.info("Detected CPU count: %d", cpu_count)
    logger.info("Current load average (1m): %.2f", load_avg)
    logger.info("Selected perm_jobs: %d (%s)", perm_jobs, perm_jobs_mode)
    if load_avg > cpu_count:
        logger.warning("System load warning: load_avg (%.2f) is greater than cpu_count (%d).", load_avg, cpu_count)
    if perm_jobs > 1:
        print("Reminder: set OMP_NUM_THREADS=1 to avoid oversubscription")

    run_mode = "unsupervised_only" if args.run_unsupervised else "full_pipeline"
    logger.info("Run mode: %s", run_mode)
    if args.run_unsupervised:
        logger.info("Output directory: %s", args.outdir / "unsupervised")
    else:
        logger.info("Output directory: %s", args.outdir)
    logger.info("Loading dataset: %s", args.input)
    df = load_dataset(args.input)
    logger.info("Loaded shape: %s", df.shape)


    drop_weather_reason = [c for c in df.columns if "weather_missing_reason" in c.lower()]
    if drop_weather_reason:
        df = df.drop(columns=drop_weather_reason)
        logger.info("Dropped weather-missing-reason columns: %s", drop_weather_reason)
        logger.info("Shape after drop: %s", df.shape)

    outcome_col = infer_outcome_column(df)
    date_col = infer_date_column(df)
    df, temporal_cols = derive_temporal_columns(df, date_col)
    df, moon_cyclic_cols = add_moon_cyclic_features(df)

    col_groups = classify_columns(df, outcome_col, temporal_cols)
    if args.run_unsupervised:
        run_unsupervised_structure(
            df=df,
            outcome_col=outcome_col,
            col_groups=col_groups,
            args=args,
            logger=logger,
        )
        logger.info("Unsupervised module completed.")
        return

    out_paths = ensure_dirs(args.outdir)

    outcome_dist = (
        df[outcome_col]
        .astype(str)
        .value_counts(dropna=False)
        .rename_axis("outcome")
        .reset_index(name="count")
    )
    outcome_dist["pct"] = (100.0 * outcome_dist["count"] / len(df)).round(2)
    outcome_dist.to_csv(out_paths["tables"] / "outcome_distribution.csv", index=False)

    feature_group_summary = pd.DataFrame(
        [
            {"group": "weather_numeric", "n_columns": len(col_groups["weather_numeric"])},
            {"group": "weather_categorical", "n_columns": len(col_groups["weather_categorical"])},
            {"group": "moon_numeric", "n_columns": len(col_groups["moon_numeric"])},
            {"group": "moon_categorical", "n_columns": len(col_groups["moon_categorical"])},
            {"group": "moon_default_model_numeric", "n_columns": len(col_groups["moon_default_numeric"])},
            {"group": "moon_threshold_model_numeric", "n_columns": len(col_groups["moon_threshold_numeric"])},
            {"group": "moon_threshold_model_categorical", "n_columns": len(col_groups["moon_threshold_categorical"])},
            {"group": "temporal_derived", "n_columns": len(temporal_cols)},
            {"group": "all_numeric_model", "n_columns": len(col_groups["all_numeric"])},
            {"group": "all_categorical_model", "n_columns": len(col_groups["all_categorical"])},
            {"group": "excluded_metadata", "n_columns": len(col_groups["metadata"])},
            {"group": "excluded_leakage", "n_columns": len(col_groups["leakage"])},
        ]
    )
    feature_group_summary.to_csv(out_paths["tables"] / "feature_group_summary.csv", index=False)




    weather_before = len(col_groups["weather_model_numeric"])
    kept_weather_numeric = correlation_filter_weather(
        df=df,
        weather_numeric_cols=col_groups["weather_model_numeric"],
        threshold=args.corr_threshold,
        out_paths=out_paths,
    )
    weather_after = len(kept_weather_numeric)
    logger.info(
        "Global correlation filtering kept %d/%d environmental numeric features (|r| > %.2f).",
        weather_after,
        weather_before,
        args.corr_threshold,
    )
    logger.info("Filtered environmental numeric feature list (kept): %s", kept_weather_numeric)

    feature_blocks = build_feature_blocks(col_groups)
    env_numeric_original = set(col_groups["weather_model_numeric"])
    feature_blocks["environmental_only"]["numeric"] = sorted(kept_weather_numeric)
    feature_blocks["combined"]["numeric"] = sorted(
        [c for c in feature_blocks["combined"]["numeric"] if c not in env_numeric_original]
        + list(kept_weather_numeric)
    )
    feature_blocks_summary = build_feature_blocks_summary(feature_blocks)
    feature_blocks_summary.to_csv(out_paths["tables"] / "feature_blocks_summary.csv", index=False)

    cv_fold_df, cv_summary_df, cv_summary_json = run_stratified_cv_evaluation(
        df=df,
        outcome_col=outcome_col,
        feature_blocks=feature_blocks,
        out_paths=out_paths,
        random_state=args.seed,
        n_splits=args.cv_folds,
    )
    cv_binary_fold_df, cv_binary_summary_df, cv_binary_summary_json, cv_binary_shap_meta = run_binary_histgb_cv(
        df=df,
        outcome_col=outcome_col,
        feature_blocks=feature_blocks,
        out_paths=out_paths,
        random_state=args.seed,
        n_splits=args.cv_folds,
        run_shap_stability=args.run_shap_stability,
    )
    perm_dist_df, perm_summary_df, perm_meta = run_envonly_permutation_test(
        df=df,
        outcome_col=outcome_col,
        feature_blocks=feature_blocks,
        cv_binary_summary_df=cv_binary_summary_df,
        out_paths=out_paths,
        random_state=args.seed,
        n_splits=args.cv_folds,
        n_perm=args.perm_n,
        perm_jobs=perm_jobs,
        run_test=args.run_permutation_test,
    )
    lgbm_cv_fold_df = pd.DataFrame()
    lgbm_cv_summary_df = pd.DataFrame()
    lgbm_cv_summary_json: Dict[str, object] = {}
    lgbm_shap_meta: Dict[str, object] = {"enabled": False, "summary_csv": "", "warnings": [], "runtime_seconds": 0.0}
    lgbm_perm_dist_df = pd.DataFrame()
    lgbm_perm_summary_df = pd.DataFrame()
    lgbm_perm_meta: Dict[str, object] = {"enabled": False, "distribution_csv": "", "summary_csv": "", "plot_files": [], "perm_jobs": perm_jobs, "runtime_seconds": 0.0}
    lgbm_compare_df = pd.DataFrame()
    if args.run_lightgbm_replication:
        if LGBMClassifier is None:
            raise ImportError("LightGBM replication requested but lightgbm is not installed. Install with: pip install lightgbm")
        logger.info("Starting LightGBM binary replication path.")
        lgbm_cv_fold_df, lgbm_cv_summary_df, lgbm_cv_summary_json, lgbm_shap_meta = run_binary_lgbm_cv(
            df=df,
            outcome_col=outcome_col,
            feature_blocks=feature_blocks,
            out_paths=out_paths,
            random_state=args.seed,
            n_splits=args.cv_folds,
        )
        lgbm_perm_dist_df, lgbm_perm_summary_df, lgbm_perm_meta = run_envonly_permutation_test_lgbm(
            df=df,
            outcome_col=outcome_col,
            feature_blocks=feature_blocks,
            cv_binary_summary_df=lgbm_cv_summary_df,
            out_paths=out_paths,
            random_state=args.seed,
            n_splits=args.cv_folds,
            n_perm=args.perm_n,
            perm_jobs=perm_jobs,
            run_test=args.run_permutation_test,
        )
        lgbm_compare_df = build_binary_model_robustness_comparison(
            hist_cv_summary_df=cv_binary_summary_df,
            hist_perm_summary_df=perm_summary_df,
            lgbm_cv_summary_df=lgbm_cv_summary_df,
            lgbm_perm_summary_df=lgbm_perm_summary_df,
        )
        compare_path = out_paths["tables"] / "binary_model_robustness_comparison.csv"
        lgbm_compare_df.to_csv(compare_path, index=False)
        logger.info("Binary robustness model comparison: %s", compare_path)
    shap_stability_path = Path(cv_binary_shap_meta["stability"]["stability_csv"])
    if shap_stability_path.exists():
        try:
            shap_stability_df = pd.read_csv(shap_stability_path)
        except pd.errors.EmptyDataError:
            shap_stability_df = pd.DataFrame()
    else:
        shap_stability_df = pd.DataFrame()
    robustness_md = build_robustness_summary_markdown(
        shap_stability_df=shap_stability_df,
        perm_summary_df=perm_summary_df,
        run_permutation_test=args.run_permutation_test,
    )
    robustness_md_path = out_paths["root"] / "robustness_summary.md"
    robustness_md_path.write_text(robustness_md, encoding="utf-8")

    logger.info("Outcome column: %s", outcome_col)
    logger.info("Temporal columns: %s", temporal_cols)
    logger.info("Moon cyclic columns added: %s", moon_cyclic_cols)
    logger.info("Weather numeric: %d", len(col_groups["weather_numeric"]))
    logger.info("Weather categorical: %d", len(col_groups["weather_categorical"]))
    logger.info("Moon columns: %d", len(col_groups["moon_numeric"]) + len(col_groups["moon_categorical"]))
    logger.info("Feature blocks saved: %s", out_paths["tables"] / "feature_blocks_summary.csv")
    logger.info("CV fold metrics rows: %d", len(cv_fold_df))
    logger.info("Binary CV fold metrics rows: %d", len(cv_binary_fold_df))
    logger.info("Binary SHAP summary: %s", cv_binary_shap_meta["summary_csv"])
    logger.info("Binary SHAP top20: %s", cv_binary_shap_meta["top20_csv"])
    logger.info("Binary SHAP plots generated: %d", len(cv_binary_shap_meta["plot_files"]))
    logger.info("Binary SHAP fold rankings: %s", cv_binary_shap_meta["stability"]["rankings_csv"])
    logger.info("Binary SHAP stability summary: %s", cv_binary_shap_meta["stability"]["stability_csv"])
    logger.info("Binary SHAP stability plots generated: %d", len(cv_binary_shap_meta["stability"]["plot_files"]))
    logger.info("Binary SHAP runtime (seconds): %.2f", cv_binary_shap_meta["runtime_seconds"])
    if cv_binary_shap_meta["warnings"]:
        for msg in cv_binary_shap_meta["warnings"]:
            logger.warning(msg)
    else:
        logger.info("Binary SHAP warnings: none")
    if perm_meta["enabled"]:
        logger.info(
            "Permutation methodology unchanged: HistGB binary model family, same features/filtering/hyperparameters, same StratifiedKFold setup."
        )
        logger.info("Permutation backend: joblib loky active=%s, perm_jobs=%d", bool(perm_jobs > 1), int(perm_jobs))
        logger.info("Permutation distribution: %s", perm_meta["distribution_csv"])
        logger.info("Permutation summary: %s", perm_meta["summary_csv"])
        logger.info("Permutation plots generated: %d", len(perm_meta["plot_files"]))
        logger.info("Permutation runtime (seconds): %.2f", perm_meta["runtime_seconds"])
    else:
        logger.info("Permutation test skipped.")
    if args.run_lightgbm_replication:
        logger.info("LightGBM binary CV fold metrics rows: %d", len(lgbm_cv_fold_df))
        logger.info("LightGBM binary CV summary: %s", out_paths["tables"] / "cv_binary_metric_summary_lgbm.csv")
        logger.info("LightGBM binary CV fold file: %s", out_paths["tables"] / "cv_binary_fold_metrics_lgbm.csv")
        if lgbm_shap_meta.get("summary_csv"):
            logger.info("LightGBM SHAP summary: %s", lgbm_shap_meta["summary_csv"])
        if lgbm_shap_meta.get("warnings"):
            for msg in lgbm_shap_meta["warnings"]:
                logger.warning(msg)
        if lgbm_perm_meta["enabled"]:
            logger.info(
                "Permutation methodology unchanged: LightGBM binary model family, same features/filtering/hyperparameters, same StratifiedKFold setup."
            )
            logger.info("LightGBM permutation backend: joblib loky active=%s, perm_jobs=%d", bool(perm_jobs > 1), int(perm_jobs))
            logger.info("LightGBM permutation distribution: %s", lgbm_perm_meta["distribution_csv"])
            logger.info("LightGBM permutation summary: %s", lgbm_perm_meta["summary_csv"])
            logger.info("LightGBM permutation plots generated: %d", len(lgbm_perm_meta["plot_files"]))
            logger.info("LightGBM permutation runtime (seconds): %.2f", lgbm_perm_meta["runtime_seconds"])
        if not lgbm_compare_df.empty:
            logger.info("HistGB vs LightGBM robustness comparison (env-only):")
            for _, row in lgbm_compare_df.iterrows():
                logger.info(
                    "%s | HistGB AUC=%.4f | LightGBM AUC=%.4f | HistGB p=%.4f | LightGBM p=%.4f | both robust agree=%s",
                    row["disease"],
                    row["hist_observed_auc"],
                    row["lgbm_observed_auc"],
                    row["hist_perm_p_value"],
                    row["lgbm_perm_p_value"],
                    "yes" if row["robustness_agreement"] == "agree" else ("no" if row["robustness_agreement"] == "disagree" else "n/a"),
                )
    logger.info("Robustness summary: %s", robustness_md_path)
    logger.info(
        "Runtime summary | CPUs detected: %d | Load average (1m): %.2f | perm_jobs used: %d | Total permutation runtime (seconds): %.2f",
        cpu_count,
        load_avg,
        perm_jobs,
        perm_meta["runtime_seconds"],
    )
    logger.info("Methodology unchanged; execution parallelized only.")


    classic_summary = save_classic_eda_block(
        df=df,
        date_col=date_col,
        weather_numeric_cols=col_groups["weather_numeric"],
        moon_numeric_cols=col_groups["moon_numeric"],
        moon_categorical_cols=col_groups["moon_categorical"],
        out_paths=out_paths,
    )


    pca_evr_df = run_pca_weather(
        df=df,
        outcome_col=outcome_col,
        weather_numeric_cols=col_groups["weather_numeric"],
        out_paths=out_paths,
    )


    rf_all_metrics, rf_all_importance = run_random_forest(
        df=df,
        outcome_col=outcome_col,
        numeric_cols=col_groups["all_numeric"],
        categorical_cols=col_groups["all_categorical"],
        out_paths=out_paths,
        run_name="rf_all_features",
        random_state=args.seed,
        test_size=args.test_size,
    )


    rf_weather_metrics, rf_weather_importance = run_random_forest(
        df=df,
        outcome_col=outcome_col,
        numeric_cols=kept_weather_numeric,
        categorical_cols=col_groups["weather_model_categorical"],
        out_paths=out_paths,
        run_name="rf_environmental_filtered",
        random_state=args.seed,
        test_size=args.test_size,
    )


    top_num = (
        rf_weather_importance[rf_weather_importance["feature"].str.startswith("num__", na=False)]
        .head(12)["feature"]
        .str.replace("num__", "", regex=False)
        .tolist()
    )
    logistic_or_df = run_logistic_odds_ratio(
        df=df,
        outcome_col=outcome_col,
        features=top_num,
        out_paths=out_paths,
        random_state=args.seed,
        bootstrap_repeats=args.or_bootstrap,
        output_csv_name="logistic_odds_ratios_weather.csv",
        output_plot_name="logistic_odds_ratio_top_features.png",
        write_outputs=True,
    )

    moon_sensitivity_df = run_moon_encoding_sensitivity(
        df=df,
        outcome_col=outcome_col,
        col_groups=col_groups,
        corr_threshold=args.corr_threshold,
        out_paths=out_paths,
        random_state=args.seed,
        test_size=args.test_size,
        bootstrap_repeats=args.or_bootstrap,
    )

    report = build_markdown_report(
        out_paths=out_paths,
        dataset_shape=df.shape,
        outcome_col=outcome_col,
        outcome_dist=outcome_dist,
        feature_group_summary=feature_group_summary,
        feature_blocks_summary=feature_blocks_summary,
        cv_summary_df=cv_summary_df,
        pca_evr_df=pca_evr_df,
        rf_all_metrics=rf_all_metrics,
        rf_weather_metrics=rf_weather_metrics,
        corr_threshold=args.corr_threshold,
        weather_before=weather_before,
        weather_after=weather_after,
        logistic_or_df=logistic_or_df,
        classic_summary=classic_summary,
        moon_sensitivity_df=moon_sensitivity_df,
    )
    (out_paths["root"] / "eda_v3_report.md").write_text(report, encoding="utf-8")

    run_meta = {
        "input_path": str(args.input),
        "output_root": str(out_paths["root"]),
        "dataset_shape": list(df.shape),
        "outcome_col": outcome_col,
        "date_col": date_col,
        "moon_cyclic_columns": moon_cyclic_cols,
        "feature_blocks": {
            k: {
                "n_numeric": len(v["numeric"]),
                "n_categorical": len(v["categorical"]),
                "n_total": len(v["numeric"]) + len(v["categorical"]),
            }
            for k, v in feature_blocks.items()
        },
        "cv_folds": args.cv_folds,
        "cv_summary": cv_summary_json,
        "cv_binary_summary": cv_binary_summary_json,
        "cv_binary_shap": cv_binary_shap_meta,
        "permutation_test": perm_meta,
        "lightgbm_replication": {
            "enabled": bool(args.run_lightgbm_replication),
            "cv_binary_summary": lgbm_cv_summary_json,
            "cv_binary_shap": lgbm_shap_meta,
            "permutation_test": lgbm_perm_meta,
            "comparison_rows": int(len(lgbm_compare_df)),
        },
        "permutation_settings": {
            "run_permutation_test": bool(args.run_permutation_test),
            "perm_n": int(args.perm_n),
            "perm_jobs": int(perm_jobs),
            "perm_jobs_mode": perm_jobs_mode,
            "cpu_count_detected": int(cpu_count),
            "load_avg_1m": float(load_avg),
        },
        "robustness_summary_path": str(robustness_md_path),
        "corr_threshold": args.corr_threshold,
        "rf_all_metrics": rf_all_metrics,
        "rf_weather_metrics": rf_weather_metrics,
        "weather_numeric_before": weather_before,
        "weather_numeric_after": weather_after,
        "moon_encoding_sensitivity_rows": int(len(moon_sensitivity_df)),
    }
    (out_paths["root"] / "run_metadata.json").write_text(json.dumps(run_meta, indent=2), encoding="utf-8")

    logger.info("EDA v3 completed.")
    logger.info("Report: %s", out_paths["root"] / "eda_v3_report.md")


if __name__ == "__main__":
    main()
